# 🍅 Tomato Crop Disease Progression Model
### Multi-Disease Temporal Predictor — GRU with Cross-Disease Attention

**Diseases tracked:** `early_blight · late_blight · leaf_mold · powdery_mildew · spider_mites`

**Predictions per sample window:**
1. Infection % at 24 h ahead (regression, per disease)
2. Infection % at 48 h ahead (regression, per disease)
3. Disease becomes active within 24 h (binary, per disease)
4. Disease becomes active within 48 h (binary, per disease)
5. Net infection change over 24 h / 48 h (regression, per disease)

**Scenarios supported:**
- **Multi-disease**: multiple diseases active simultaneously on the same crop
- **Single-disease**: exactly one disease progressing
- **Healthy**: no disease present (all infection values near zero)

**Architecture:** Improved Multi-stream GRU with Cross-Disease Co-infection Attention

---

## SECTION 0 — Run ID & Path Setup

In [3]:
# ─────────────────────────────────────────────────────────────
# SECTION 0: Run ID & Centralised Path Setup
# ─────────────────────────────────────────────────────────────

import os
import logging
from datetime import datetime

# ── Generate a unique run ID for this execution ───────────────
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Run ID : {RUN_ID}')

# ── Root project path ─────────────────────────────────────────
PROJECT_ROOT = r'E:\AgriTwin-GH'

# ── Dataset path ──────────────────────────────────────────────
DATA_PATH = os.path.join(
    PROJECT_ROOT, 'data', 'processed', 'Disease Progression',
    'tomato_disease_progression_synthetic_hourly.csv'
)

# ── Model save path (same artifact structure as growth stage notebook) ─
MODEL_DIR  = os.path.join(PROJECT_ROOT, 'src', 'agritwin_gh', 'models')
MODEL_PATH = os.path.join(MODEL_DIR, f'disease_progression_{RUN_ID}.keras')

# ── Artifacts sub-structure (mirrors growth stage notebook) ──
ARTIFACTS_DIR = os.path.join(MODEL_DIR, 'artifacts', f'disease_progression_{RUN_ID}')
PLOTS_DIR     = ARTIFACTS_DIR
METRICS_DIR   = ARTIFACTS_DIR
LOGS_DIR      = ARTIFACTS_DIR
REPORTS_DIR   = ARTIFACTS_DIR

# ── Checkpoint path (used during training) ────────────────────
CKPT_PATH = os.path.join(ARTIFACTS_DIR, f'best_model_{RUN_ID}.keras')

# ── Create all directories ────────────────────────────────────
for d in [MODEL_DIR, ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Run log setup ─────────────────────────────────────────────
LOG_FILE = os.path.join(LOGS_DIR, f'run_{RUN_ID}.log')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger('disease_progression')
logger.info(f'Run started | run_id={RUN_ID}')

print(f'Dataset         : {DATA_PATH}')
print(f'Model save path : {MODEL_PATH}')
print(f'Artifacts dir   : {ARTIFACTS_DIR}')
print(f'Log file        : {LOG_FILE}')
print('\n✅ Paths configured.')


2026-03-12 14:18:42,450 | INFO | Run started | run_id=20260312_141842


Run ID : 20260312_141842
Dataset         : E:\AgriTwin-GH\data\processed\Disease Progression\tomato_disease_progression_synthetic_hourly.csv
Model save path : E:\AgriTwin-GH\src\agritwin_gh\models\disease_progression_20260312_141842.keras
Artifacts dir   : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842
Log file        : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\run_20260312_141842.log

✅ Paths configured.


## SECTION 1 — Setup: Libraries, Seeds, Version Info

In [4]:
# ─────────────────────────────────────────────────────────────
# SECTION 1: Setup
# ─────────────────────────────────────────────────────────────

import sys
import json
import pickle
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from collections import Counter
from pathlib import Path

# Sklearn
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, mean_absolute_error,
    mean_squared_error, roc_auc_score, ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,
)

# Optional SHAP
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('SHAP not available — skipping SHAP plots.')

# ── Reproducibility ──────────────────────────────────────────
RANDOM_SEED = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100


def save_fig(fig, name: str, subdir: str = '') -> str:
    """Save *fig* to PLOTS_DIR (optionally a subdir) and return path."""
    target_dir = os.path.join(PLOTS_DIR, subdir) if subdir else PLOTS_DIR
    os.makedirs(target_dir, exist_ok=True)
    path = os.path.join(target_dir, f'{name}.png')
    fig.savefig(path, bbox_inches='tight', dpi=150)
    return path


# ── Version info ─────────────────────────────────────────────
print(f'Python  : {sys.version}')
print(f'TF      : {tf.__version__}')
print(f'Keras   : {keras.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'Pandas  : {pd.__version__}')
print(f'GPU     : {tf.config.list_physical_devices("GPU")}')
logger.info(f'TF={tf.__version__}  GPU={tf.config.list_physical_devices("GPU")}')
print('\n✅ Setup complete.')


e:\AgriTwin-GH\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-12 14:20:34,815 | INFO | TF=2.20.0  GPU=[]


Python  : 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]
TF      : 2.20.0
Keras   : 3.13.2
NumPy   : 2.4.2
Pandas  : 3.0.1
GPU     : []

✅ Setup complete.


## SECTION 2 — Data Loading

In [5]:
# ─────────────────────────────────────────────────────────────
# SECTION 2: Data Loading
# ─────────────────────────────────────────────────────────────

print(f'Loading dataset from:\n  {DATA_PATH}')
assert os.path.exists(DATA_PATH), f'Dataset not found: {DATA_PATH}'
raw_df = pd.read_csv(DATA_PATH)
logger.info(f'Dataset loaded | shape={raw_df.shape} | path={DATA_PATH}')

# ── Quick overview ────────────────────────────────────────────
print(f'\nShape   : {raw_df.shape}')
print(f'Columns : {list(raw_df.columns)}')
print('\n── Head (first 10 rows) ──')
display(raw_df.head(10))

print('\n── Info ──')
raw_df.info()

print('\n── Missing values ──')
missing = raw_df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values.')

print('\n── Descriptive statistics (numeric) ──')
display(raw_df.describe())

print('\n── Unique diseases in dataset ──')
print(raw_df['disease_name'].value_counts())

print('\n── Unique crop cycles ──')
print(f'  cycle_id unique values: {sorted(raw_df["cycle_id"].unique())}')
print(f'  Rows per cycle (first disease only):')
print(raw_df[raw_df['disease_name'] == raw_df['disease_name'].iloc[0]].groupby('cycle_id').size())

print('\n── Disease present flag distribution ──')
print(raw_df['disease_present_flag'].value_counts())

# ── Save dataset summary to artifacts ────────────────────────
dataset_summary = {
    'dataset_path'   : DATA_PATH,
    'n_rows'         : int(raw_df.shape[0]),
    'n_cols'         : int(raw_df.shape[1]),
    'columns'        : list(raw_df.columns),
    'dtypes'         : {c: str(t) for c, t in raw_df.dtypes.items()},
    'missing_values' : raw_df.isnull().sum().to_dict(),
    'diseases'       : raw_df['disease_name'].unique().tolist(),
    'n_cycles'       : int(raw_df['cycle_id'].nunique()),
    'memory_usage_mb': round(raw_df.memory_usage(deep=True).sum() / 1e6, 3),
}
with open(os.path.join(METRICS_DIR, 'dataset_summary.json'), 'w') as f:
    json.dump(dataset_summary, f, indent=2, default=str)
print(f'\n✅ Dataset summary saved.')


Loading dataset from:
  E:\AgriTwin-GH\data\processed\Disease Progression\tomato_disease_progression_synthetic_hourly.csv


2026-03-12 14:20:42,050 | INFO | Dataset loaded | shape=(54240, 39) | path=E:\AgriTwin-GH\data\processed\Disease Progression\tomato_disease_progression_synthetic_hourly.csv



Shape   : (54240, 39)
Columns : ['timestamp', 'cycle_id', 'cycle_label', 'season_label', 'stage_name', 'stage_index', 'days_from_cycle_start', 'day_of_year', 'week_of_year', 'hour', 'hours_in_current_stage', 'stage_progress_pct', 'total_cycle_progress_pct', 'is_stage_transition', 'indoor_temp', 'indoor_humidity', 'indoor_air_velocity', 'indoor_CO2', 'solarradiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy', 'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h', 'vpd_proxy', 'cumulative_gdd_like_index', 'disease_name', 'disease_present_flag', 'disease_cycle_id', 'disease_cycle_stage', 'outbreak_trigger_flag', 'control_action_flag', 'control_action_type', 'stage_susceptibility_score', 'disease_risk_score', 'hours_since_disease_onset', 'current_infection_pct', 'infection_growth_rate_hourly']

── Head (first 10 rows) ──


,timestamp,cycle_id,cycle_label,season_label,stage_name,stage_index,days_from_cycle_start,day_of_year,week_of_year,hour,hours_in_current_stage,stage_progress_pct,total_cycle_progress_pct,is_stage_transition,indoor_temp,indoor_humidity,indoor_air_velocity,indoor_CO2,solarradiation,day_night_flag,vpd,dew_point,leaf_wetness_proxy,temperature_rolling_mean_24h,humidity_rolling_mean_24h,vpd_proxy,cumulative_gdd_like_index,disease_name,disease_present_flag,disease_cycle_id,disease_cycle_stage,outbreak_trigger_flag,control_action_flag,control_action_type,stage_susceptibility_score,disease_risk_score,hours_since_disease_onset,current_infection_pct,infection_growth_rate_hourly
0,2024-07-01 00:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0000,183,27,0,0.0,0.000,0.000,False,29.002223,75.929629,2.41,440.0,0.0,0.0,0.964305,24.188148,0.0,29.0022,75.9296,0.9643,0.7918,early_blight,0,0,none,0,0,none,0.35,0.3500,NaN,0.0,0.0
1,2024-07-01 00:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0000,183,27,0,0.0,0.000,0.000,False,29.002223,75.929629,2.41,440.0,0.0,0.0,0.964305,24.188148,0.0,29.0022,75.9296,0.9643,0.7918,late_blight,0,0,none,0,0,none,0.45,0.1007,NaN,0.0,0.0
2,2024-07-01 00:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0000,183,27,0,0.0,0.000,0.000,False,29.002223,75.929629,2.41,440.0,0.0,0.0,0.964305,24.188148,0.0,29.0022,75.9296,0.9643,0.7918,leaf_mold,0,0,none,0,0,none,0.30,0.0817,NaN,0.0,0.0
3,2024-07-01 00:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0000,183,27,0,0.0,0.000,0.000,False,29.002223,75.929629,2.41,440.0,0.0,0.0,0.964305,24.188148,0.0,29.0022,75.9296,0.9643,0.7918,powdery_mildew,0,0,none,0,0,none,0.50,0.4125,NaN,0.0,0.0
4,2024-07-01 00:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0000,183,27,0,0.0,0.000,0.000,False,29.002223,75.929629,2.41,440.0,0.0,0.0,0.964305,24.188148,0.0,29.0022,75.9296,0.9643,0.7918,spider_mites,0,0,none,0,0,none,0.40,0.1893,NaN,0.0,0.0
5,2024-07-01 01:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0417,183,27,1,1.0,0.347,0.037,False,27.801924,70.830127,2.41,400.0,0.0,1.0,1.089948,21.967949,0.0,28.4021,73.3799,1.0899,1.5335,early_blight,0,0,none,0,0,none,0.35,0.3500,NaN,0.0,0.0
6,2024-07-01 01:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0417,183,27,1,1.0,0.347,0.037,False,27.801924,70.830127,2.41,400.0,0.0,1.0,1.089948,21.967949,0.0,28.4021,73.3799,1.0899,1.5335,late_blight,0,0,none,0,0,none,0.45,0.0661,NaN,0.0,0.0
7,2024-07-01 01:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0417,183,27,1,1.0,0.347,0.037,False,27.801924,70.830127,2.41,400.0,0.0,1.0,1.089948,21.967949,0.0,28.4021,73.3799,1.0899,1.5335,leaf_mold,0,0,none,0,0,none,0.30,0.0646,NaN,0.0,0.0
8,2024-07-01 01:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0417,183,27,1,1.0,0.347,0.037,False,27.801924,70.830127,2.41,400.0,0.0,1.0,1.089948,21.967949,0.0,28.4021,73.3799,1.0899,1.5335,powdery_mildew,0,0,none,0,0,none,0.50,0.4914,NaN,0.0,0.0
9,2024-07-01 01:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0417,183,27,1,1.0,0.347,0.037,False,27.801924,70.830127,2.41,400.0,0.0,1.0,1.089948,21.967949,0.0,28.4021,73.3799,1.0899,1.5335,spider_mites,0,0,none,0,0,none,0.40,0.2009,NaN,0.0,0.0



── Info ──
<class 'pandas.DataFrame'>
RangeIndex: 54240 entries, 0 to 54239
Data columns (total 39 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   timestamp                     54240 non-null  str    
 1   cycle_id                      54240 non-null  int64  
 2   cycle_label                   54240 non-null  str    
 3   season_label                  54240 non-null  str    
 4   stage_name                    54240 non-null  str    
 5   stage_index                   54240 non-null  int64  
 6   days_from_cycle_start         54240 non-null  float64
 7   day_of_year                   54240 non-null  int64  
 8   week_of_year                  54240 non-null  int64  
 9   hour                          54240 non-null  int64  
 10  hours_in_current_stage        54240 non-null  float64
 11  stage_progress_pct            54240 non-null  float64
 12  total_cycle_progress_pct      54240 non-null  float64
 13  

,cycle_id,stage_index,days_from_cycle_start,day_of_year,week_of_year,hour,hours_in_current_stage,stage_progress_pct,total_cycle_progress_pct,indoor_temp,indoor_humidity,indoor_air_velocity,indoor_CO2,solarradiation,day_night_flag,vpd,dew_point,leaf_wetness_proxy,temperature_rolling_mean_24h,humidity_rolling_mean_24h,vpd_proxy,cumulative_gdd_like_index,disease_present_flag,disease_cycle_id,outbreak_trigger_flag,control_action_flag,stage_susceptibility_score,disease_risk_score,hours_since_disease_onset,current_infection_pct,infection_growth_rate_hourly
count,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.00000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,5.424000e+04,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,54240.000000,18880.000000,54240.000000,54240.000000
mean,2.508850,2.595133,56.514565,193.659292,28.046460,11.50000,262.756637,49.889381,49.981563,30.631134,69.722829,2.167367,417.530457,6.740770e+01,0.509864,1.445034,24.575700,0.072824,30.630590,69.682592,1.445033,1066.504295,0.348083,1.160785,0.000553,0.152212,0.671757,0.352811,376.027066,11.713011,-0.000511
std,1.118009,1.660933,32.681824,95.508300,13.583856,6.92225,187.971043,28.867732,28.867778,3.799954,11.419702,0.578638,22.284203,8.771010e+01,0.499907,0.808442,2.570191,0.259851,2.203336,9.747405,0.808443,621.497264,0.476367,1.866923,0.023512,0.359230,0.174458,0.252354,267.684004,24.338652,0.735656
min,1.000000,0.000000,0.000000,1.000000,1.000000,0.00000,0.000000,0.000000,0.000000,19.801924,38.320522,0.940000,383.860000,0.000000e+00,0.000000,0.000000,17.325786,0.000000,23.505300,47.499900,0.000000,0.675100,0.000000,0.000000,0.000000,0.000000,0.300000,0.000000,0.000000,0.000000,-8.924927
25%,1.750000,1.000000,28.239575,119.750000,17.750000,5.75000,112.750000,24.968500,24.991000,27.934361,61.305905,1.840000,397.574361,0.000000e+00,0.000000,0.832236,22.742362,0.000000,28.960875,61.184725,0.832250,532.505875,0.000000,0.000000,0.000000,0.000000,0.550000,0.132600,161.000000,0.000000,0.000000
50%,3.000000,3.000000,56.479150,208.000000,30.000000,11.50000,226.000000,49.937000,49.982000,30.200000,69.688281,2.230000,400.000000,1.181784e-15,1.000000,1.321468,24.348148,0.000000,31.026400,70.004950,1.321450,1058.777700,0.000000,0.000000,0.000000,0.000000,0.680000,0.348550,324.000000,0.000000,0.000000
75%,3.250000,4.000000,84.718725,264.250000,38.000000,17.25000,374.000000,74.905500,74.973000,33.100000,78.100000,2.590000,440.000000,1.338500e+02,1.000000,1.883551,26.360091,0.000000,32.453575,77.207675,1.883525,1591.492950,1.000000,2.000000,0.000000,0.000000,0.820000,0.550000,537.000000,7.318650,0.000000
max,4.000000,5.000000,116.958300,366.000000,52.000000,23.00000,791.000000,99.874000,99.964000,41.167777,100.000000,4.070000,440.000000,3.228000e+02,1.000000,4.756544,32.285052,1.000000,34.756900,95.325300,4.756500,2270.187000,1.000000,6.000000,1.000000,1.000000,0.920000,0.920000,1099.000000,100.000000,7.440790



── Unique diseases in dataset ──
disease_name
early_blight      10848
late_blight       10848
leaf_mold         10848
powdery_mildew    10848
spider_mites      10848
Name: count, dtype: int64

── Unique crop cycles ──
  cycle_id unique values: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  Rows per cycle (first disease only):
cycle_id
1    2712
2    2616
3    2808
4    2712
dtype: int64

── Disease present flag distribution ──
disease_present_flag
0    35360
1    18880
Name: count, dtype: int64

✅ Dataset summary saved.


## SECTION 3 — Data Standardization & Preprocessing

In [7]:

# ─────────────────────────────────────────────────────────────
# SECTION 3: Data Standardization & Preprocessing
# ─────────────────────────────────────────────────────────────

# ── 3.1  Disease registry ─────────────────────────────────────
DISEASES = [
    'early_blight',
    'late_blight',
    'leaf_mold',
    'powdery_mildew',
    'spider_mites',
]
DISEASE_TO_INT = {d: i for i, d in enumerate(DISEASES)}
INT_TO_DISEASE = {i: d for d, i in DISEASE_TO_INT.items()}
N_DISEASES = len(DISEASES)

# ── 3.2  Growth stage registry (inherited from backbone data) ──
STAGE_ORDER = [
    'seedling', 'early_vegetative', 'flowering_initiation',
    'flowering', 'unripe', 'ripe',
]
STAGE_TO_INT = {s: i for i, s in enumerate(STAGE_ORDER)}
INT_TO_STAGE = {i: s for s, i in STAGE_TO_INT.items()}
N_STAGES = len(STAGE_ORDER)

print('Disease mapping :', DISEASE_TO_INT)
print('Stage mapping   :', STAGE_TO_INT)

# ── 3.3  Column standardisation ───────────────────────────────
ALIAS_MAP = {
    'time': 'timestamp', 'datetime': 'timestamp', 'date_time': 'timestamp',
    'recorded_at': 'timestamp', 'ts': 'timestamp',
    'cycle': 'cycle_id', 'batch_id': 'cycle_id', 'crop_cycle': 'cycle_id',
    'stage_name': 'stage', 'growth_stage': 'stage', 'phenological_stage': 'stage',
    'disease': 'disease_name', 'pathogen': 'disease_name',
    'indoor_temp': 'temperature', 'air_temp': 'temperature',
    'indoor_humidity': 'humidity', 'rh': 'humidity',
    'indoor_CO2': 'co2', 'co2_ppm': 'co2',
    'indoor_air_velocity': 'air_velocity',
    'infection_pct': 'current_infection_pct',
}


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    df.rename(columns={k: v for k, v in ALIAS_MAP.items() if k in df.columns}, inplace=True)
    return df


def detect_required_columns(df: pd.DataFrame) -> dict:
    required = ['timestamp', 'cycle_id', 'stage', 'disease_name',
                'disease_present_flag', 'current_infection_pct']
    detected, missing = {}, []
    for col in required:
        if col in df.columns:
            detected[col] = col
        else:
            missing.append(col)
    if missing:
        raise ValueError(
            f'Missing required columns after alias mapping: {missing}\n'
            f'Available columns: {list(df.columns)}'
        )
    optional = ['temperature', 'humidity', 'co2', 'air_velocity', 'solarradiation',
                'vpd', 'dew_point', 'leaf_wetness_proxy', 'disease_risk_score',
                'hours_since_disease_onset', 'infection_growth_rate_hourly',
                'stage_susceptibility_score', 'outbreak_trigger_flag',
                'control_action_flag', 'is_stage_transition',
                'cumulative_gdd_like_index', 'stage_progress_pct',
                'total_cycle_progress_pct', 'day_night_flag']
    for col in optional:
        if col in df.columns:
            detected[col] = col
    print('\n── Detected columns ──')
    for k, v in detected.items():
        print(f'  {k:35s} → {v}')
    extra = [c for c in df.columns if c not in list(detected.values())]
    print(f'  Extra / unlisted columns: {extra}')
    return detected


# ── 3.4  Apply standardisation ────────────────────────────────
df = standardize_columns(raw_df)
detected_cols = detect_required_columns(df)

# ── 3.5  Parse timestamp & sort ──────────────────────────────
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.sort_values(['cycle_id', 'disease_name', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

# ── 3.6  Encode disease name ───────────────────────────────────
# Normalise spelling variants
df['disease_name'] = (
    df['disease_name'].astype(str).str.strip().str.lower()
    .str.replace(r'\s+', '_', regex=True)
)
unknown_diseases = set(df['disease_name'].unique()) - set(DISEASES)
if unknown_diseases:
    print(f'\n⚠️  Unknown diseases found: {unknown_diseases}  → dropping rows')
    df = df[df['disease_name'].isin(DISEASES)].copy()
df['disease_int'] = df['disease_name'].map(DISEASE_TO_INT).astype(int)

# ── 3.7  Encode stage name ────────────────────────────────────
df['stage_int'] = df['stage'].map(STAGE_TO_INT)
invalid_stage = df['stage_int'].isna()
if invalid_stage.sum() > 0:
    print(f'\n⚠️  {invalid_stage.sum()} rows with unrecognised stage labels — dropping')
    df = df[~invalid_stage].copy()
df['stage_int'] = df['stage_int'].astype(int)

# ── 3.8  Handle missing values ───────────────────────────────
n_before = len(df)

# Ensure sort order is correct for time-series ffill
df.sort_values(['cycle_id', 'disease_name', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

# Identify numeric columns AFTER encoding (so disease_int / stage_int are included)
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

# Forward-fill then back-fill within each (cycle_id, disease_name) group.
# Using transform() avoids the pandas 2.x groupby.apply() index-mutation issue
# where group keys can end up as index levels and get dropped by reset_index(drop=True).
df[numeric_cols] = (
    df.groupby(['cycle_id', 'disease_name'], sort=False)[numeric_cols]
    .transform(lambda x: x.ffill().bfill())
)

# Fill any residual NaNs with 0 for genuinely absent records
df[numeric_cols] = df[numeric_cols].fillna(0)

n_dups_before = df.duplicated(subset=['cycle_id', 'disease_name', 'timestamp']).sum()
if n_dups_before > 0:
    print(f'\n⚠️  Removing {n_dups_before} duplicate (cycle, disease, timestamp) rows')
    df = df.drop_duplicates(subset=['cycle_id', 'disease_name', 'timestamp'], keep='last')

df.sort_values(['cycle_id', 'disease_name', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'\nRows before cleaning : {n_before}')
print(f'Rows after  cleaning : {len(df)}')
print(f'\nData after standardisation: {df.shape}')
display(df.head(10))
logger.info(f'Standardisation complete | shape={df.shape}')


Disease mapping : {'early_blight': 0, 'late_blight': 1, 'leaf_mold': 2, 'powdery_mildew': 3, 'spider_mites': 4}
Stage mapping   : {'seedling': 0, 'early_vegetative': 1, 'flowering_initiation': 2, 'flowering': 3, 'unripe': 4, 'ripe': 5}

── Detected columns ──
  timestamp                           → timestamp
  cycle_id                            → cycle_id
  stage                               → stage
  disease_name                        → disease_name
  disease_present_flag                → disease_present_flag
  current_infection_pct               → current_infection_pct
  temperature                         → temperature
  humidity                            → humidity
  air_velocity                        → air_velocity
  solarradiation                      → solarradiation
  vpd                                 → vpd
  dew_point                           → dew_point
  leaf_wetness_proxy                  → leaf_wetness_proxy
  disease_risk_score                  → disease_risk_scor

,timestamp,cycle_id,cycle_label,season_label,stage,stage_index,days_from_cycle_start,day_of_year,week_of_year,hour,hours_in_current_stage,stage_progress_pct,total_cycle_progress_pct,is_stage_transition,temperature,humidity,air_velocity,indoor_co2,solarradiation,day_night_flag,vpd,dew_point,leaf_wetness_proxy,temperature_rolling_mean_24h,humidity_rolling_mean_24h,vpd_proxy,cumulative_gdd_like_index,disease_name,disease_present_flag,disease_cycle_id,disease_cycle_stage,outbreak_trigger_flag,control_action_flag,control_action_type,stage_susceptibility_score,disease_risk_score,hours_since_disease_onset,current_infection_pct,infection_growth_rate_hourly,disease_int,stage_int
0,2024-07-01 00:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0000,183,27,0,0.0,0.000,0.000,False,29.002223,75.929629,2.41,440.000000,0.000000,0.0,0.964305,24.188148,0.0,29.0022,75.9296,0.9643,0.7918,early_blight,0,0,none,0,0,none,0.35,0.3500,0.0,0.0,0.0,0,0
1,2024-07-01 01:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0417,183,27,1,1.0,0.347,0.037,False,27.801924,70.830127,2.41,400.000000,0.000000,1.0,1.089948,21.967949,0.0,28.4021,73.3799,1.0899,1.5335,early_blight,0,0,none,0,0,none,0.35,0.3500,0.0,0.0,0.0,0,0
2,2024-07-01 02:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.0833,183,27,2,2.0,0.694,0.074,False,28.278680,70.035534,2.41,400.000000,0.000000,1.0,1.151145,22.285786,0.0,28.3609,72.2651,1.1511,2.2951,early_blight,0,0,none,0,0,none,0.35,0.3500,0.0,0.0,0.0,0,0
3,2024-07-01 03:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.1250,183,27,3,3.0,1.042,0.111,False,28.900000,69.000000,2.41,400.000000,0.000000,1.0,1.234602,22.700000,0.0,28.4957,71.4488,1.2346,3.0826,early_blight,0,0,none,0,0,none,0.35,0.3500,0.0,0.0,0.0,0,0
4,2024-07-01 04:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.1667,183,27,4,4.0,1.389,0.147,False,29.623543,67.794095,2.41,400.000000,0.000000,1.0,1.337286,23.182362,0.0,28.7213,70.7179,1.3373,3.9003,early_blight,0,0,none,0,0,none,0.35,0.3500,0.0,0.0,0.0,0,0
5,2024-07-01 05:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.2083,183,27,5,5.0,1.736,0.184,False,30.400000,66.500000,2.41,400.000000,0.000000,1.0,1.454366,23.700000,0.0,29.0011,70.0149,1.4544,4.7336,early_blight,0,0,none,0,0,none,0.35,0.3500,0.0,0.0,0.0,0,0
6,2024-07-01 06:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.2500,183,27,6,6.0,2.083,0.221,False,31.176457,65.205905,2.41,400.000000,0.000000,1.0,1.578929,24.217638,0.0,29.3118,69.3279,1.5789,5.5669,early_blight,0,0,none,0,0,none,0.35,0.3500,0.0,0.0,0.0,0,0
7,2024-07-01 07:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.2917,183,27,7,7.0,2.431,0.258,False,32.820361,63.539820,2.41,397.699099,46.018026,1.0,1.815602,25.528324,0.0,29.7504,68.6044,1.8156,6.4003,early_blight,0,0,none,0,0,none,0.35,0.3309,0.0,0.0,0.0,0,0
8,2024-07-01 08:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.3333,183,27,8,8.0,2.778,0.295,False,34.299320,62.075466,2.41,395.555000,88.900000,1.0,2.051169,26.714414,0.0,30.2558,67.8790,2.0512,7.2336,early_blight,0,0,none,0,0,none,0.35,0.2934,0.0,0.0,0.0,0,0
9,2024-07-01 09:00:00,1,kharif_2024,southwest_monsoon,seedling,0,0.3750,183,27,9,9.0,3.125,0.332,False,35.512548,60.912637,2.41,393.713821,125.723586,1.0,2.260788,27.695075,0.0,30.7815,67.1823,2.2608,8.0669,early_blight,0,0,none,0,0,none,0.35,0.2528,0.0,0.0,0.0,0,0


2026-03-12 14:24:01,768 | INFO | Standardisation complete | shape=(54240, 41)


## SECTION 4 — Disease Cycle Integrity Checks

In [8]:
# ─────────────────────────────────────────────────────────────
# SECTION 4: Disease Cycle Integrity Checks
# ─────────────────────────────────────────────────────────────
#
# A "disease series" is the unique time series for a single
# (cycle_id, disease_name) pair.  We mirror the cycle check
# logic from the growth stage notebook, adapted for disease data.
# ─────────────────────────────────────────────────────────────

MIN_ROWS_PER_SERIES = 24   # need at least 24 h of data to form one sequence


def check_disease_series(grp: pd.DataFrame) -> dict:
    grp = grp.sort_values('timestamp')
    cid     = grp['cycle_id'].iloc[0]
    disease = grp['disease_name'].iloc[0]
    n_rows  = len(grp)
    start_time   = grp['timestamp'].min()
    end_time     = grp['timestamp'].max()
    duration_h   = (end_time - start_time).total_seconds() / 3600
    n_active     = int(grp['disease_present_flag'].sum())
    peak_inf     = float(grp['current_infection_pct'].max())
    n_outbreaks  = int(grp['outbreak_trigger_flag'].sum()) if 'outbreak_trigger_flag' in grp else 0

    issues = []
    if not grp['timestamp'].is_monotonic_increasing:
        issues.append('non_monotonic_timestamps')
    n_dups = grp['timestamp'].duplicated().sum()
    if n_dups > 0:
        issues.append(f'{int(n_dups)}_duplicate_timestamps')
    if n_rows < MIN_ROWS_PER_SERIES:
        issues.append(f'only_{n_rows}_rows')

    # Check for unrealistic infection values
    if (grp['current_infection_pct'] < 0).any():
        issues.append('negative_infection_pct')
    if (grp['current_infection_pct'] > 100).any():
        issues.append('infection_pct_exceeds_100')

    return {
        'cycle_id'      : cid,
        'disease_name'  : disease,
        'n_rows'        : n_rows,
        'n_active_hours': n_active,
        'peak_infection': round(peak_inf, 2),
        'n_outbreaks'   : n_outbreaks,
        'duration_hours': round(duration_h, 1),
        'start_time'    : start_time,
        'end_time'      : end_time,
        'issues'        : '; '.join(issues) if issues else 'OK',
    }


print('Checking disease series integrity...')
series_records = [
    check_disease_series(grp)
    for _, grp in df.groupby(['cycle_id', 'disease_name'], sort=False)
]
series_summary = pd.DataFrame(series_records)

print(f'\nTotal disease series : {len(series_summary)}')
print(f'  = {df["cycle_id"].nunique()} cycles × {N_DISEASES} diseases')
print('\n── Series Summary (head) ──')
display(series_summary.head(20))

print('\n── Issues breakdown ──')
issue_mask = series_summary['issues'] != 'OK'
print(f'  OK          : {(~issue_mask).sum()}')
print(f'  With issues : {issue_mask.sum()}')
if issue_mask.sum() > 0:
    display(series_summary[issue_mask])

# ── 4.2  Remove hard-error series ────────────────────────────
HARD_KEYWORDS = ['only_', 'negative_infection']

def is_hard_error(s: str) -> bool:
    return any(kw in s for kw in HARD_KEYWORDS)

hard_bad = series_summary.loc[
    series_summary['issues'].apply(is_hard_error),
    ['cycle_id', 'disease_name']
]
print(f'\nSeries removed (hard errors): {len(hard_bad)}')

if len(hard_bad) > 0:
    bad_keys = set(zip(hard_bad['cycle_id'], hard_bad['disease_name']))
    df = df[~df.apply(lambda r: (r['cycle_id'], r['disease_name']) in bad_keys, axis=1)].copy()

df.sort_values(['cycle_id', 'disease_name', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Data after integrity cleaning : {df.shape}')
logger.info(f'Integrity check done | total_series={len(series_summary)} hard_removed={len(hard_bad)}')

# ── 4.3  Save series summary ──────────────────────────────────
series_summary_path = os.path.join(METRICS_DIR, 'disease_series_summary.csv')
series_summary.to_csv(series_summary_path, index=False)
print(f'Series summary saved → {series_summary_path}')


Checking disease series integrity...

Total disease series : 20
  = 4 cycles × 5 diseases

── Series Summary (head) ──


,cycle_id,disease_name,n_rows,n_active_hours,peak_infection,n_outbreaks,duration_hours,start_time,end_time,issues
0,1,early_blight,2712,1345,91.37,2,2711.0,2024-07-01,2024-10-21 23:00:00,OK
1,1,late_blight,2712,962,87.30,1,2711.0,2024-07-01,2024-10-21 23:00:00,OK
2,1,leaf_mold,2712,1317,84.56,2,2711.0,2024-07-01,2024-10-21 23:00:00,OK
3,1,powdery_mildew,2712,1271,100.00,2,2711.0,2024-07-01,2024-10-21 23:00:00,OK
4,1,spider_mites,2712,1301,88.18,2,2711.0,2024-07-01,2024-10-21 23:00:00,OK
5,2,early_blight,2616,530,40.24,1,2615.0,2024-11-10,2025-02-26 23:00:00,OK
6,2,late_blight,2616,903,47.25,2,2615.0,2024-11-10,2025-02-26 23:00:00,OK
7,2,leaf_mold,2616,601,46.47,1,2615.0,2024-11-10,2025-02-26 23:00:00,OK
8,2,powdery_mildew,2616,770,95.19,1,2615.0,2024-11-10,2025-02-26 23:00:00,OK
9,2,spider_mites,2616,542,44.55,1,2615.0,2024-11-10,2025-02-26 23:00:00,OK


2026-03-12 14:24:06,008 | INFO | Integrity check done | total_series=20 hard_removed=0



── Issues breakdown ──
  OK          : 20
  With issues : 0

Series removed (hard errors): 0
Data after integrity cleaning : (54240, 41)
Series summary saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\disease_series_summary.csv


## SECTION 5 — Progression Target Construction (24h & 48h Look-ahead)

In [9]:
# ─────────────────────────────────────────────────────────────
# SECTION 5: Progression Target Construction
# ─────────────────────────────────────────────────────────────
#
# For each row in a (cycle_id, disease_name) time series, derive:
#
#   infection_pct_24h   — actual infection % exactly 24 timesteps ahead
#   infection_pct_48h   — actual infection % exactly 48 timesteps ahead
#   active_within_24h   — binary: disease_present_flag=1 at 24-h horizon
#   active_within_48h   — binary: disease_present_flag=1 at 48-h horizon
#   net_change_24h      — regression: Δinfection over 24 h (can be negative)
#   net_change_48h      — regression: Δinfection over 48 h
#
# Data is hourly, so 24 steps = 24 h, 48 steps = 48 h.
# Rows within the final 48 hours of a series will have NaN for 48h targets
# and NaN for 24h targets in the last 24 hours — handled via sample weights.
# ─────────────────────────────────────────────────────────────

HORIZON_24 = 24   # timesteps
HORIZON_48 = 48   # timesteps


def build_disease_targets(series_df: pd.DataFrame) -> pd.DataFrame:
    """
    For a single (cycle_id, disease_name) series, compute look-ahead targets.
    Rows without a future horizon (at end of series) receive NaN.
    """
    grp = series_df.sort_values('timestamp').reset_index(drop=True)
    n   = len(grp)

    inf_pct    = grp['current_infection_pct'].values.astype(np.float32)
    active_flag= grp['disease_present_flag'].values.astype(np.float32)

    inf_24  = np.full(n, np.nan, dtype=np.float32)
    inf_48  = np.full(n, np.nan, dtype=np.float32)
    act_24  = np.full(n, np.nan, dtype=np.float32)
    act_48  = np.full(n, np.nan, dtype=np.float32)
    dlt_24  = np.full(n, np.nan, dtype=np.float32)
    dlt_48  = np.full(n, np.nan, dtype=np.float32)

    for i in range(n):
        if i + HORIZON_24 < n:
            inf_24[i] = inf_pct[i + HORIZON_24]
            act_24[i] = active_flag[i + HORIZON_24]
            dlt_24[i] = inf_pct[i + HORIZON_24] - inf_pct[i]
        if i + HORIZON_48 < n:
            inf_48[i] = inf_pct[i + HORIZON_48]
            act_48[i] = active_flag[i + HORIZON_48]
            dlt_48[i] = inf_pct[i + HORIZON_48] - inf_pct[i]

    grp['infection_pct_24h']   = inf_24
    grp['infection_pct_48h']   = inf_48
    grp['active_within_24h']   = act_24
    grp['active_within_48h']   = act_48
    grp['net_change_24h']      = dlt_24
    grp['net_change_48h']      = dlt_48
    return grp


print('Building progression targets (24h & 48h look-ahead)...')
target_chunks = [
    build_disease_targets(grp)
    for _, grp in df.groupby(['cycle_id', 'disease_name'], sort=False)
]
df = pd.concat(target_chunks, ignore_index=True)
df.sort_values(['cycle_id', 'disease_name', 'timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)

# ── Verification ──────────────────────────────────────────────
target_cols_check = [
    'cycle_id', 'disease_name', 'timestamp', 'current_infection_pct',
    'disease_present_flag',
    'infection_pct_24h', 'infection_pct_48h',
    'active_within_24h', 'active_within_48h',
    'net_change_24h', 'net_change_48h',
]
print(f'\nTargets built.  Shape: {df.shape}')
print('\nSample target values (first disease in first cycle):')
sample_series = df[(df['cycle_id'] == df['cycle_id'].iloc[0]) &
                   (df['disease_name'] == df['disease_name'].iloc[0])]
display(sample_series[target_cols_check].head(20))

print('\n── Target coverage (non-NaN rows) ──')
for col in ['infection_pct_24h', 'infection_pct_48h',
            'active_within_24h', 'active_within_48h',
            'net_change_24h',    'net_change_48h']:
    valid = df[col].notna().sum()
    print(f'  {col:25s}: {valid:6d} / {len(df)} ({100*valid/len(df):.1f}%)')

print('\n── Target statistics ──')
display(df[['infection_pct_24h','infection_pct_48h',
            'net_change_24h','net_change_48h']].describe())

logger.info(f'Targets built | shape={df.shape}')


Building progression targets (24h & 48h look-ahead)...

Targets built.  Shape: (54240, 47)

Sample target values (first disease in first cycle):


,cycle_id,disease_name,timestamp,current_infection_pct,disease_present_flag,infection_pct_24h,infection_pct_48h,active_within_24h,active_within_48h,net_change_24h,net_change_48h
0,1,early_blight,2024-07-01 00:00:00,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,early_blight,2024-07-01 01:00:00,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,early_blight,2024-07-01 02:00:00,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,early_blight,2024-07-01 03:00:00,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,early_blight,2024-07-01 04:00:00,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
5,1,early_blight,2024-07-01 05:00:00,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
6,1,early_blight,2024-07-01 06:00:00,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
7,1,early_blight,2024-07-01 07:00:00,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
8,1,early_blight,2024-07-01 08:00:00,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0
9,1,early_blight,2024-07-01 09:00:00,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0



── Target coverage (non-NaN rows) ──
  infection_pct_24h        :  53760 / 54240 (99.1%)
  infection_pct_48h        :  53280 / 54240 (98.2%)
  active_within_24h        :  53760 / 54240 (99.1%)
  active_within_48h        :  53280 / 54240 (98.2%)
  net_change_24h           :  53760 / 54240 (99.1%)
  net_change_48h           :  53280 / 54240 (98.2%)

── Target statistics ──


,infection_pct_24h,infection_pct_48h,net_change_24h,net_change_48h
count,53760.000000,53280.000000,53760.000000,53280.000000
mean,11.751576,11.800685,-0.007061,-0.004427
std,24.367283,24.419655,5.063663,8.403424
min,0.000000,0.000000,-58.844402,-79.018600
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,7.421800,7.574050,0.000000,0.000000
max,100.000000,100.000000,41.229496,60.900600


2026-03-12 14:24:15,997 | INFO | Targets built | shape=(54240, 47)


## SECTION 6 — Exploratory Data Analysis

In [10]:
# ─────────────────────────────────────────────────────────────
# SECTION 6: Exploratory Data Analysis
# ─────────────────────────────────────────────────────────────

# ── 6.1  Disease presence distribution ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Active hours per disease
active_per_disease = (
    df.groupby('disease_name')['disease_present_flag']
    .sum()
    .reindex(DISEASES)
)
active_per_disease.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2', N_DISEASES))
axes[0].set_title('Total Active (Infected) Hours per Disease')
axes[0].set_xlabel('Disease')
axes[0].set_ylabel('Hours Active')
axes[0].tick_params(axis='x', rotation=30)

# Peak infection % distribution per disease
df.boxplot(column='current_infection_pct', by='disease_name',
           ax=axes[1], grid=False, figsize=(7, 4))
axes[1].set_title('Infection % Distribution by Disease')
axes[1].set_xlabel('Disease')
axes[1].set_ylabel('Infection Percentage (%)')
axes[1].tick_params(axis='x', rotation=30)
plt.suptitle('')
plt.tight_layout()
p = save_fig(fig, 'disease_presence_distribution')
print(f'Disease presence chart saved → {p}')
plt.show(); plt.close(fig)

# ── 6.2  Infection progression timelines for sample cycles ───
sample_cycles = sorted(df['cycle_id'].unique())[:3]
palette = sns.color_palette('tab10', N_DISEASES)
disease_colors = {d: palette[i] for i, d in enumerate(DISEASES)}

fig, axes = plt.subplots(len(sample_cycles), 1, figsize=(16, 4 * len(sample_cycles)))
if len(sample_cycles) == 1:
    axes = [axes]

for ax, cid in zip(axes, sample_cycles):
    cdf = df[df['cycle_id'] == cid].sort_values('timestamp')
    for disease in DISEASES:
        dsub = cdf[cdf['disease_name'] == disease]
        if dsub.empty:
            continue
        ax.plot(dsub['timestamp'], dsub['current_infection_pct'],
                label=disease, color=disease_colors[disease], linewidth=1.5, alpha=0.85)
    ax.set_title(f'Cycle {cid} — Disease Infection Progression')
    ax.set_xlabel('Time')
    ax.set_ylabel('Infection %')
    ax.legend(loc='upper right', fontsize=8, ncol=3)

plt.suptitle('Multi-Disease Co-infection Timelines')
plt.tight_layout()
p = save_fig(fig, 'infection_progression_timelines')
print(f'Timelines saved → {p}')
plt.show(); plt.close(fig)

# ── 6.3  Co-occurrence analysis (hourly simultaneous active diseases) ──
# Build a per-timestamp matrix of active diseases
pivot_active = df.pivot_table(
    index=['cycle_id', 'timestamp'],
    columns='disease_name',
    values='disease_present_flag',
    aggfunc='max',
).fillna(0)

# Number of simultaneously active diseases per timestamp
pivot_active['n_active'] = pivot_active[DISEASES].sum(axis=1)

print('\n── Simultaneous disease co-occurrence distribution ──')
co_occ = pivot_active['n_active'].value_counts().sort_index()
print(co_occ)

fig, ax = plt.subplots(figsize=(8, 4))
co_occ.plot(kind='bar', ax=ax, color=sns.color_palette('Blues_d', len(co_occ)))
ax.set_title('Distribution of Simultaneously Active Disease Count per Timestep')
ax.set_xlabel('Number of Active Diseases')
ax.set_ylabel('Number of Hourly Records')
plt.tight_layout()
p = save_fig(fig, 'disease_cooccurrence_distribution')
print(f'Co-occurrence chart saved → {p}')
plt.show(); plt.close(fig)

# ── 6.4  Disease × Stage susceptibility heatmap ──────────────
susc_data = df.groupby(['disease_name', 'stage'])['stage_susceptibility_score'].mean().unstack()
susc_data = susc_data.reindex(DISEASES)
susc_data = susc_data.reindex(columns=STAGE_ORDER[:len(susc_data.columns)])

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(susc_data, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Susceptibility Score'})
ax.set_title('Disease Stage Susceptibility (mean per disease × growth stage)')
ax.set_xlabel('Growth Stage')
ax.set_ylabel('Disease')
plt.tight_layout()
p = save_fig(fig, 'disease_stage_susceptibility_heatmap')
print(f'Susceptibility heatmap saved → {p}')
plt.show(); plt.close(fig)

# ── 6.5  Environmental correlation with infection growth rate ─
env_corr_cols = ['temperature', 'humidity', 'co2', 'air_velocity',
                 'vpd', 'dew_point', 'leaf_wetness_proxy',
                 'disease_risk_score', 'infection_growth_rate_hourly',
                 'current_infection_pct']
env_corr_cols = [c for c in env_corr_cols if c in df.columns]
corr_mat = df[env_corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, mask=mask, linewidths=0.3, vmin=-1, vmax=1)
ax.set_title('Environmental Feature Correlation Matrix')
plt.tight_layout()
p = save_fig(fig, 'environmental_correlation_matrix')
print(f'Correlation heatmap saved → {p}')
plt.show(); plt.close(fig)

# ── 6.6  Target distribution ─────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
target_plot_cols = [
    ('infection_pct_24h', 'Infection % at 24h', 'steelblue'),
    ('infection_pct_48h', 'Infection % at 48h', 'darkorange'),
    ('net_change_24h',    'Net Change at 24h',  'seagreen'),
    ('net_change_48h',    'Net Change at 48h',  'crimson'),
    ('active_within_24h', 'Active at 24h (binary)', 'mediumpurple'),
    ('active_within_48h', 'Active at 48h (binary)', 'goldenrod'),
]
for ax, (col, title, color) in zip(axes.flat, target_plot_cols):
    valid_vals = df[col].dropna()
    ax.hist(valid_vals, bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.suptitle('Target Variable Distributions')
plt.tight_layout()
p = save_fig(fig, 'target_distributions')
print(f'Target distributions saved → {p}')
plt.show(); plt.close(fig)

logger.info('EDA visualizations complete.')
print('\n✅ EDA complete.')


Disease presence chart saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\disease_presence_distribution.png
Timelines saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\infection_progression_timelines.png

── Simultaneous disease co-occurrence distribution ──
n_active
0    2116
1    3167
2    2601
3    1517
4    1275
5     172
Name: count, dtype: int64
Co-occurrence chart saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\disease_cooccurrence_distribution.png
Susceptibility heatmap saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\disease_stage_susceptibility_heatmap.png
Correlation heatmap saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\environmental_correlation_matrix.png


2026-03-12 14:24:34,989 | INFO | EDA visualizations complete.


Target distributions saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\target_distributions.png

✅ EDA complete.


## SECTION 7 — Feature Engineering

In [11]:
# ─────────────────────────────────────────────────────────────
# SECTION 7: Feature Engineering
# ─────────────────────────────────────────────────────────────
#
# Feature categories:
#   A. Time-based features (shared across diseases)
#   B. Environmental rolling stats & lags (shared)
#   C. Disease-specific temporal features (per disease series)
#   D. Crop-level features (stage, growth progress)
#   E. Multi-disease interaction features (aggregated across diseases)
# ─────────────────────────────────────────────────────────────

# ── 7.1  Raw environmental columns available ─────────────────
ENV_COLS = [c for c in ['temperature', 'humidity', 'co2', 'air_velocity',
                         'solarradiation', 'vpd', 'dew_point', 'leaf_wetness_proxy',
                         'cumulative_gdd_like_index', 'day_night_flag',
                         'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h',
                         'vpd_proxy']
            if c in df.columns]

DISEASE_COLS = [c for c in ['current_infection_pct', 'disease_present_flag',
                              'disease_risk_score', 'hours_since_disease_onset',
                              'infection_growth_rate_hourly', 'stage_susceptibility_score',
                              'outbreak_trigger_flag', 'control_action_flag']
                if c in df.columns]

print(f'Environmental columns  : {ENV_COLS}')
print(f'Disease columns        : {DISEASE_COLS}')

# ── 7.2  Feature engineering per (cycle_id, disease_name) series ─
ROLLING_WINDOWS = [6, 12, 24]
LAG_STEPS       = [1, 2, 3, 6, 12, 24]


def engineer_disease_series_features(grp: pd.DataFrame) -> pd.DataFrame:
    """
    Derive features for a single (cycle_id, disease_name) time series.
    All rolling/lag operations are strictly within the series to prevent
    data leakage across cycles or diseases.
    """
    grp = grp.sort_values('timestamp').copy()
    cid     = grp['cycle_id'].iloc[0]
    disease = grp['disease_name'].iloc[0]

    # ── A. Time-based features ────────────────────────────────
    t0 = grp['timestamp'].min()
    grp['elapsed_hours']    = (grp['timestamp'] - t0).dt.total_seconds() / 3600.0
    grp['hour_of_day']      = grp['timestamp'].dt.hour
    grp['day_of_week']      = grp['timestamp'].dt.dayofweek
    grp['day_of_cycle']     = (grp['elapsed_hours'] / 24).astype(int)
    grp['week_of_cycle']    = (grp['elapsed_hours'] / 168).astype(int)
    # Cyclical encoding of hour
    grp['hour_sin']         = np.sin(2 * np.pi * grp['hour_of_day'] / 24.0)
    grp['hour_cos']         = np.cos(2 * np.pi * grp['hour_of_day'] / 24.0)

    # ── B. Environmental rolling stats & lags ─────────────────
    for col in ENV_COLS:
        if col not in grp.columns:
            continue
        for w in ROLLING_WINDOWS:
            grp[f'{col}_roll_mean_{w}h'] = grp[col].rolling(w, min_periods=1).mean()
            grp[f'{col}_roll_std_{w}h']  = grp[col].rolling(w, min_periods=1).std().fillna(0)
        for lag in LAG_STEPS:
            grp[f'{col}_lag_{lag}'] = grp[col].shift(lag)

    # ── C. Disease-specific temporal features ─────────────────
    inf_col = 'current_infection_pct'
    if inf_col in grp.columns:
        for w in ROLLING_WINDOWS:
            grp[f'infection_roll_mean_{w}h'] = grp[inf_col].rolling(w, min_periods=1).mean()
            grp[f'infection_roll_std_{w}h']  = grp[inf_col].rolling(w, min_periods=1).std().fillna(0)
            grp[f'infection_roll_max_{w}h']  = grp[inf_col].rolling(w, min_periods=1).max()
        for lag in LAG_STEPS:
            grp[f'infection_lag_{lag}'] = grp[inf_col].shift(lag)

        # Cumulative infection load
        grp['cumulative_infection']  = grp[inf_col].expanding().sum()
        # Rate of change (first difference)
        grp['infection_delta_1h']    = grp[inf_col].diff(1).fillna(0)
        grp['infection_delta_6h']    = grp[inf_col].diff(6).fillna(0)
        grp['infection_delta_24h']   = grp[inf_col].diff(24).fillna(0)
        # Peak infection seen so far (running max — no target leakage)
        grp['infection_running_max'] = grp[inf_col].expanding().max()

    if 'hours_since_disease_onset' in grp.columns:
        grp['hours_since_onset_lag_1'] = grp['hours_since_disease_onset'].shift(1)

    if 'infection_growth_rate_hourly' in grp.columns:
        for w in [6, 12]:
            grp[f'growth_rate_roll_mean_{w}h'] = grp['infection_growth_rate_hourly'].rolling(w, min_periods=1).mean()

    # ── D. Disease identity encoding ─────────────────────────
    # One-hot encode the disease within the feature vector
    for d in DISEASES:
        grp[f'is_{d}'] = int(disease == d)

    # ── E. Control action encoding ───────────────────────────
    if 'control_action_flag' in grp.columns:
        grp['consecutive_control_hours'] = (
            grp['control_action_flag']
            .groupby((grp['control_action_flag'] != grp['control_action_flag'].shift()).cumsum())
            .cumcount()
        )

    return grp


print('Engineering features per (cycle_id, disease_name) series...')
print(f'  Total series: {df.groupby(["cycle_id","disease_name"]).ngroups}')

feat_chunks = [
    engineer_disease_series_features(grp)
    for _, grp in df.groupby(['cycle_id', 'disease_name'], sort=False)
]
df_feat = pd.concat(feat_chunks, ignore_index=True)
df_feat.sort_values(['cycle_id', 'disease_name', 'timestamp'], inplace=True)
df_feat.reset_index(drop=True, inplace=True)

# ── 7.3  Identify all feature columns ─────────────────────────
TARGET_COLS = {
    'infection_pct_24h', 'infection_pct_48h',
    'active_within_24h', 'active_within_48h',
    'net_change_24h', 'net_change_48h',
}
NON_FEATURE_COLS = {
    'timestamp', 'cycle_id', 'cycle_label', 'season_label',
    'disease_name', 'disease_cycle_id', 'disease_cycle_stage',
    'control_action_type', 'stage',
} | TARGET_COLS

FEATURE_COLS = [
    c for c in df_feat.columns
    if c not in NON_FEATURE_COLS
    and pd.api.types.is_numeric_dtype(df_feat[c])
]
print(f'\nTotal features: {len(FEATURE_COLS)}')
print('Features:', FEATURE_COLS[:30], '...' if len(FEATURE_COLS) > 30 else '')

# ── 7.4  Back-fill NaN from lags at series start ─────────────
df_feat[FEATURE_COLS] = df_feat[FEATURE_COLS].bfill().fillna(0)

print(f'\nFinal feature-engineered DataFrame shape: {df_feat.shape}')
display(df_feat[FEATURE_COLS[:6]].head(5))
logger.info(f'Feature engineering done | n_features={len(FEATURE_COLS)} shape={df_feat.shape}')

# ── Save feature list ──────────────────────────────────────────
feature_info = {
    'n_features'     : len(FEATURE_COLS),
    'feature_cols'   : FEATURE_COLS,
    'rolling_windows': ROLLING_WINDOWS,
    'lag_steps'      : LAG_STEPS,
    'env_cols'       : ENV_COLS,
    'disease_cols'   : DISEASE_COLS,
    'target_cols'    : list(TARGET_COLS),
}
with open(os.path.join(METRICS_DIR, 'feature_list.json'), 'w') as f:
    json.dump(feature_info, f, indent=2)
print('Feature list saved.')


Environmental columns  : ['temperature', 'humidity', 'air_velocity', 'solarradiation', 'vpd', 'dew_point', 'leaf_wetness_proxy', 'cumulative_gdd_like_index', 'day_night_flag', 'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h', 'vpd_proxy']
Disease columns        : ['current_infection_pct', 'disease_present_flag', 'disease_risk_score', 'hours_since_disease_onset', 'infection_growth_rate_hourly', 'stage_susceptibility_score', 'outbreak_trigger_flag', 'control_action_flag']
Engineering features per (cycle_id, disease_name) series...
  Total series: 20

Total features: 212
Features: ['stage_index', 'days_from_cycle_start', 'day_of_year', 'week_of_year', 'hour', 'hours_in_current_stage', 'stage_progress_pct', 'total_cycle_progress_pct', 'is_stage_transition', 'temperature', 'humidity', 'air_velocity', 'indoor_co2', 'solarradiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy', 'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h', 'vpd_proxy', 'cumulative_gdd_

,stage_index,days_from_cycle_start,day_of_year,week_of_year,hour,hours_in_current_stage
0,0,0.0000,183,27,0,0.0
1,0,0.0417,183,27,1,1.0
2,0,0.0833,183,27,2,2.0
3,0,0.1250,183,27,3,3.0
4,0,0.1667,183,27,4,4.0


2026-03-12 14:24:47,009 | INFO | Feature engineering done | n_features=212 shape=(54240, 227)


Feature list saved.


## SECTION 8 — Scenario-Aware Wide-Format Dataset Construction

The dataset is currently in **long format** (one row per `cycle_id × timestamp × disease`).

To enable the model to observe **all diseases simultaneously** on the same crop at each timestep,
we **pivot to wide format**: one row per `(cycle_id, timestamp)` with per-disease feature columns.

This wide-format representation allows:
- Cross-disease interaction learning (co-infection patterns)
- Healthy-crop rows (all diseases at zero) to be naturally included
- Scenario labelling (multi-disease / single-disease / healthy)

In [13]:

# ─────────────────────────────────────────────────────────────
# SECTION 8: Scenario-Aware Wide-Format Dataset Construction
# ─────────────────────────────────────────────────────────────

# ── 8.1  Separate shared (env) vs disease-specific features ──
#
# Shared features: environmental + time-based (same for all diseases in a row)
# Disease-specific features: per-disease infection, risk, susceptibility, etc.
# ─────────────────────────────────────────────────────────────

# Features that are NOT disease-specific (one value per timestamp per cycle)
SHARED_FEATURE_PREFIXES = [
    'temperature', 'humidity', 'co2', 'air_velocity', 'solarradiation',
    'vpd', 'dew_point', 'leaf_wetness_proxy', 'cumulative_gdd',
    'day_night_flag', 'hour_of_day', 'day_of_week', 'elapsed_hours',
    'day_of_cycle', 'week_of_cycle', 'hour_sin', 'hour_cos',
    'stage_int', 'stage_progress_pct', 'total_cycle_progress_pct',
    'is_stage_transition', 'days_from_cycle_start', 'day_of_year',
    'week_of_year', 'hour',
]

def is_shared_feature(col: str) -> bool:
    if col.startswith('is_'):
        return False   # disease one-hot flags are disease-specific
    for prefix in SHARED_FEATURE_PREFIXES:
        if col.startswith(prefix):
            return True
    return False

shared_cols  = [c for c in FEATURE_COLS if is_shared_feature(c)]
disease_cols = [c for c in FEATURE_COLS if not is_shared_feature(c)]

print(f'Shared features         : {len(shared_cols)}')
print(f'Disease-specific feats  : {len(disease_cols)}')

# ── 8.2  Build target columns per disease (wide format) ──────
TARGET_PREFIXES = [
    'infection_pct_24h', 'infection_pct_48h',
    'active_within_24h', 'active_within_48h',
    'net_change_24h',    'net_change_48h',
]

# ── 8.3  Pivot to wide format ─────────────────────────────────
# Take one representative row per (cycle_id, timestamp) — use first disease
# for the shared features (they are identical across diseases)
shared_base = (
    df_feat[df_feat['disease_name'] == DISEASES[0]]
    [['cycle_id', 'timestamp'] + shared_cols]
    .drop_duplicates(subset=['cycle_id', 'timestamp'])
    .sort_values(['cycle_id', 'timestamp'])
    .reset_index(drop=True)
)

print(f'\nShared base shape: {shared_base.shape}')

# Pivot disease-specific features + targets
disease_specific_cols = disease_cols + list(TARGET_PREFIXES)
# Keep only those that exist
disease_specific_cols = [c for c in disease_specific_cols if c in df_feat.columns]

pivot_frames = []
for disease in DISEASES:
    sub = (
        df_feat[df_feat['disease_name'] == disease]
        [['cycle_id', 'timestamp'] + disease_specific_cols]
        .copy()
    )
    sub = sub.rename(columns={c: f'{disease}__{c}' for c in disease_specific_cols})
    pivot_frames.append(sub)

# Merge all disease pivots onto shared base
wide_df = shared_base.copy()
for pf in pivot_frames:
    wide_df = wide_df.merge(pf, on=['cycle_id', 'timestamp'], how='left')

wide_df = wide_df.sort_values(['cycle_id', 'timestamp']).reset_index(drop=True)

print(f'Wide-format shape        : {wide_df.shape}')
print(f'Columns: {list(wide_df.columns)[:20]} ...')

# ── 8.4  Scenario labelling ───────────────────────────────────
# Count how many diseases are simultaneously active at each timestep
active_cols_wide = [f'{d}__disease_present_flag' for d in DISEASES
                    if f'{d}__disease_present_flag' in wide_df.columns]
if active_cols_wide:
    wide_df['n_active_diseases'] = wide_df[active_cols_wide].fillna(0).sum(axis=1).astype(int)
else:
    wide_df['n_active_diseases'] = 0


def assign_scenario(n_active: int) -> str:
    if n_active == 0:
        return 'healthy'
    elif n_active == 1:
        return 'single_disease'
    else:
        return 'multi_disease'

wide_df['scenario'] = wide_df['n_active_diseases'].apply(assign_scenario)

print('\n── Scenario distribution (rows) ──')
print(wide_df['scenario'].value_counts())

fig, ax = plt.subplots(figsize=(7, 4))
wide_df['scenario'].value_counts().plot(kind='bar', ax=ax,
                                         color=['#4CAF50', '#2196F3', '#F44336'])
ax.set_title('Row Count by Scenario Type')
ax.set_xlabel('Scenario')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
p = save_fig(fig, 'scenario_distribution')
print(f'Scenario distribution saved → {p}')
plt.show(); plt.close(fig)

# ── 8.5  Collect all feature columns in wide format ──────────
# Build the full candidate list (shared first, then disease-specific wide cols),
# then deduplicate while preserving order so the shared_cols take priority.
# Deduplication is critical: shared_cols are already present in wide_df, so
# a naive concatenation would create duplicate keys that break __setitem__.
_NON_FEATURE = {'cycle_id', 'timestamp', 'scenario', 'n_active_diseases'}
_target_suffix_set = set(TARGET_PREFIXES)   # used for endswith check

_all_wide_numeric = [
    c for c in wide_df.columns
    if c not in _NON_FEATURE
    and not any(c.endswith(f'__{t}') for t in _target_suffix_set)
    and pd.api.types.is_numeric_dtype(wide_df[c])
]

# Shared cols first, remaining wide cols after — deduplicated (preserving order)
_seen: set = set()
WIDE_FEATURE_COLS: list = []
for c in (shared_cols + _all_wide_numeric):
    if c not in _seen and c in wide_df.columns:
        _seen.add(c)
        WIDE_FEATURE_COLS.append(c)

# Wide-format targets per disease
WIDE_TARGET_COLS = {}
for disease in DISEASES:
    WIDE_TARGET_COLS[disease] = {
        'inf_24h'    : f'{disease}__infection_pct_24h',
        'inf_48h'    : f'{disease}__infection_pct_48h',
        'active_24h' : f'{disease}__active_within_24h',
        'active_48h' : f'{disease}__active_within_48h',
        'delta_24h'  : f'{disease}__net_change_24h',
        'delta_48h'  : f'{disease}__net_change_48h',
    }
    # Keep only those actually present
    WIDE_TARGET_COLS[disease] = {
        k: v for k, v in WIDE_TARGET_COLS[disease].items()
        if v in wide_df.columns
    }

# Fill any residual NaN in feature cols (no duplicates → assignment is safe)
wide_df[WIDE_FEATURE_COLS] = wide_df[WIDE_FEATURE_COLS].fillna(0)

print(f'\nFinal wide-format features: {len(WIDE_FEATURE_COLS)}')
print(f'Sample row (feature names):', WIDE_FEATURE_COLS[:10])

logger.info(f'Wide format built | shape={wide_df.shape} features={len(WIDE_FEATURE_COLS)}')
print('\n✅ Wide-format dataset constructed.')


Shared features         : 173
Disease-specific feats  : 39

Shared base shape: (10848, 175)
Wide-format shape        : (10848, 400)
Columns: ['cycle_id', 'timestamp', 'days_from_cycle_start', 'day_of_year', 'week_of_year', 'hour', 'hours_in_current_stage', 'stage_progress_pct', 'total_cycle_progress_pct', 'temperature', 'humidity', 'air_velocity', 'solarradiation', 'day_night_flag', 'vpd', 'dew_point', 'leaf_wetness_proxy', 'temperature_rolling_mean_24h', 'humidity_rolling_mean_24h', 'vpd_proxy'] ...

── Scenario distribution (rows) ──
scenario
multi_disease     5565
single_disease    3167
healthy           2116
Name: count, dtype: int64


2026-03-12 14:27:54,313 | INFO | Wide format built | shape=(10848, 402) features=368


Scenario distribution saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\scenario_distribution.png

Final wide-format features: 368
Sample row (feature names): ['days_from_cycle_start', 'day_of_year', 'week_of_year', 'hour', 'hours_in_current_stage', 'stage_progress_pct', 'total_cycle_progress_pct', 'temperature', 'humidity', 'air_velocity']

✅ Wide-format dataset constructed.


## SECTION 9 — Sequence Building for LSTM / GRU

In [14]:
# ─────────────────────────────────────────────────────────────
# SECTION 9: Sequence Building
# ─────────────────────────────────────────────────────────────
#
# Sliding window over the wide-format time series (one row per
# cycle × timestamp).  The last timestep of each window provides
# the targets for all 5 diseases simultaneously.
#
# Target tensor layout per sample:
#   y_inf_24h   : (N_DISEASES,)  — regression
#   y_inf_48h   : (N_DISEASES,)  — regression
#   y_act_24h   : (N_DISEASES,)  — binary
#   y_act_48h   : (N_DISEASES,)  — binary
#   y_dlt_24h   : (N_DISEASES,)  — regression (signed delta)
#   y_dlt_48h   : (N_DISEASES,)  — regression (signed delta)
# ─────────────────────────────────────────────────────────────

SEQ_LEN = 24   # 24-hour look-back window (mirrors growth stage notebook)


def build_sequences_for_cycle(
    cycle_df: pd.DataFrame,
    feature_cols: list,
    target_cols_by_disease: dict,
    diseases: list,
    seq_len: int,
) -> tuple | None:
    """
    Slide a window of `seq_len` rows over a single crop cycle (wide format).
    Returns (X, targets_dict) or None if the cycle is too short.
    """
    cdf = cycle_df.sort_values('timestamp').reset_index(drop=True)
    n   = len(cdf)
    if n < seq_len:
        return None

    feat_vals = cdf[feature_cols].values.astype(np.float32)

    # Collect target arrays for each disease
    target_arrays = {}
    for disease in diseases:
        dc = target_cols_by_disease.get(disease, {})
        target_arrays[disease] = {
            'inf_24h' : cdf[dc['inf_24h']].values.astype(np.float32)  if 'inf_24h'  in dc else np.zeros(n, np.float32),
            'inf_48h' : cdf[dc['inf_48h']].values.astype(np.float32)  if 'inf_48h'  in dc else np.zeros(n, np.float32),
            'act_24h' : cdf[dc['active_24h']].values.astype(np.float32) if 'active_24h' in dc else np.zeros(n, np.float32),
            'act_48h' : cdf[dc['active_48h']].values.astype(np.float32) if 'active_48h' in dc else np.zeros(n, np.float32),
            'dlt_24h' : cdf[dc['delta_24h']].values.astype(np.float32) if 'delta_24h'  in dc else np.zeros(n, np.float32),
            'dlt_48h' : cdf[dc['delta_48h']].values.astype(np.float32) if 'delta_48h'  in dc else np.zeros(n, np.float32),
        }

    scenario_vals = cdf['scenario'].values

    X_list      = []
    y_inf24     = []
    y_inf48     = []
    y_act24     = []
    y_act48     = []
    y_dlt24     = []
    y_dlt48     = []
    scenarios   = []

    for end in range(seq_len - 1, n):
        start = end - seq_len + 1
        X_list.append(feat_vals[start: end + 1])

        # Stack per-disease targets into vectors of length N_DISEASES
        y_inf24.append(np.array([target_arrays[d]['inf_24h'][end] for d in diseases]))
        y_inf48.append(np.array([target_arrays[d]['inf_48h'][end] for d in diseases]))
        y_act24.append(np.array([target_arrays[d]['act_24h'][end] for d in diseases]))
        y_act48.append(np.array([target_arrays[d]['act_48h'][end] for d in diseases]))
        y_dlt24.append(np.array([target_arrays[d]['dlt_24h'][end] for d in diseases]))
        y_dlt48.append(np.array([target_arrays[d]['dlt_48h'][end] for d in diseases]))
        scenarios.append(scenario_vals[end])

    return (
        np.stack(X_list),
        np.stack(y_inf24, 0).astype(np.float32),
        np.stack(y_inf48, 0).astype(np.float32),
        np.stack(y_act24, 0).astype(np.float32),
        np.stack(y_act48, 0).astype(np.float32),
        np.stack(y_dlt24, 0).astype(np.float32),
        np.stack(y_dlt48, 0).astype(np.float32),
        np.array(scenarios),
    )


def build_all_sequences(df_wide, feature_cols, target_cols_by_disease, diseases, seq_len):
    Xs, y24s, y48s, ya24s, ya48s, yd24s, yd48s, scens = [], [], [], [], [], [], [], []
    cycle_ids_out = []

    for cid, grp in df_wide.groupby('cycle_id', sort=False):
        result = build_sequences_for_cycle(
            grp, feature_cols, target_cols_by_disease, diseases, seq_len
        )
        if result is None:
            continue
        X, y24, y48, ya24, ya48, yd24, yd48, sc = result
        Xs.append(X);      y24s.append(y24);   y48s.append(y48)
        ya24s.append(ya24); ya48s.append(ya48); yd24s.append(yd24); yd48s.append(yd48)
        scens.append(sc)
        cycle_ids_out.extend([cid] * len(y24))

    return (
        np.concatenate(Xs),
        np.concatenate(y24s),
        np.concatenate(y48s),
        np.concatenate(ya24s),
        np.concatenate(ya48s),
        np.concatenate(yd24s),
        np.concatenate(yd48s),
        np.concatenate(scens),
        np.array(cycle_ids_out),
    )


print(f'Building sequences (window={SEQ_LEN}) over wide-format data...')
(
    X_all,
    y_inf24_all, y_inf48_all,
    y_act24_all, y_act48_all,
    y_dlt24_all, y_dlt48_all,
    scenario_all, cycle_ids_all,
) = build_all_sequences(wide_df, WIDE_FEATURE_COLS, WIDE_TARGET_COLS, DISEASES, SEQ_LEN)

print(f'\nTensor shapes:')
print(f'  X                        : {X_all.shape}   [samples, timesteps, features]')
print(f'  y_infection_pct_24h      : {y_inf24_all.shape}   [samples, N_diseases]')
print(f'  y_infection_pct_48h      : {y_inf48_all.shape}')
print(f'  y_active_within_24h      : {y_act24_all.shape}')
print(f'  y_active_within_48h      : {y_act48_all.shape}')
print(f'  y_net_change_24h         : {y_dlt24_all.shape}')
print(f'  y_net_change_48h         : {y_dlt48_all.shape}')
print(f'\nScenario breakdown:')
for sc in ['healthy', 'single_disease', 'multi_disease']:
    cnt = (scenario_all == sc).sum()
    print(f'  {sc:15s}: {cnt:7d}  ({100*cnt/len(scenario_all):.1f}%)')

logger.info(f'Sequences built | total_samples={X_all.shape[0]} seq_len={SEQ_LEN} n_features={X_all.shape[2]}')


Building sequences (window=24) over wide-format data...


2026-03-12 14:28:02,570 | INFO | Sequences built | total_samples=10756 seq_len=24 n_features=368



Tensor shapes:
  X                        : (10756, 24, 368)   [samples, timesteps, features]
  y_infection_pct_24h      : (10756, 5)   [samples, N_diseases]
  y_infection_pct_48h      : (10756, 5)
  y_active_within_24h      : (10756, 5)
  y_active_within_48h      : (10756, 5)
  y_net_change_24h         : (10756, 5)
  y_net_change_48h         : (10756, 5)

Scenario breakdown:
  healthy        :    2070  (19.2%)
  single_disease :    3144  (29.2%)
  multi_disease  :    5542  (51.5%)


## SECTION 10 — Train / Validation / Test Split by Cycle

In [15]:
# ─────────────────────────────────────────────────────────────
# SECTION 10: Train / Val / Test Split  (cycle-level, no leakage)
# ─────────────────────────────────────────────────────────────
#
# Split at the cycle level to prevent temporal data leakage:
# sequences from the same cycle never span train / val / test.
# ─────────────────────────────────────────────────────────────

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# TEST_FRAC  = 0.15  (remainder)

unique_cycles = np.array(sorted(wide_df['cycle_id'].unique()))
rng = np.random.default_rng(RANDOM_SEED)
rng.shuffle(unique_cycles)

n_cycles = len(unique_cycles)
n_train  = max(1, int(n_cycles * TRAIN_FRAC))
n_val    = max(1, int(n_cycles * VAL_FRAC))
n_test   = n_cycles - n_train - n_val
if n_test < 1:
    n_val  -= 1
    n_test  = 1

train_cycles = set(unique_cycles[:n_train])
val_cycles   = set(unique_cycles[n_train: n_train + n_val])
test_cycles  = set(unique_cycles[n_train + n_val:])

print(f'Total cycles : {n_cycles}')
print(f'Train cycles : {len(train_cycles)}  → {sorted(train_cycles)}')
print(f'Val   cycles : {len(val_cycles)}    → {sorted(val_cycles)}')
print(f'Test  cycles : {len(test_cycles)}   → {sorted(test_cycles)}')

train_mask = np.isin(cycle_ids_all, list(train_cycles))
val_mask   = np.isin(cycle_ids_all, list(val_cycles))
test_mask  = np.isin(cycle_ids_all, list(test_cycles))


def apply_mask(mask):
    return (
        X_all[mask],
        y_inf24_all[mask], y_inf48_all[mask],
        y_act24_all[mask], y_act48_all[mask],
        y_dlt24_all[mask], y_dlt48_all[mask],
        scenario_all[mask],
    )


(X_tr, y_inf24_tr, y_inf48_tr,
 y_act24_tr, y_act48_tr, y_dlt24_tr, y_dlt48_tr, sc_tr) = apply_mask(train_mask)

(X_va, y_inf24_va, y_inf48_va,
 y_act24_va, y_act48_va, y_dlt24_va, y_dlt48_va, sc_va) = apply_mask(val_mask)

(X_te, y_inf24_te, y_inf48_te,
 y_act24_te, y_act48_te, y_dlt24_te, y_dlt48_te, sc_te) = apply_mask(test_mask)

print(f'\nSamples per split:')
print(f'  Train : {X_tr.shape[0]}  |  Val : {X_va.shape[0]}  |  Test : {X_te.shape[0]}')

# ── Scenario distribution per split ──────────────────────────
for split_name, sc_arr in [('Train', sc_tr), ('Val', sc_va), ('Test', sc_te)]:
    counts = {s: int((sc_arr == s).sum()) for s in ['healthy', 'single_disease', 'multi_disease']}
    print(f'  {split_name} scenarios: {counts}')

logger.info(
    f'Split | train={X_tr.shape[0]} val={X_va.shape[0]} test={X_te.shape[0]} '
    f'train_cycles={len(train_cycles)} val_cycles={len(val_cycles)} test_cycles={len(test_cycles)}'
)

# ── Save split summary ────────────────────────────────────────
split_summary = {
    'train_frac'     : TRAIN_FRAC,
    'val_frac'       : VAL_FRAC,
    'n_total_cycles' : n_cycles,
    'train_cycles'   : sorted([str(c) for c in train_cycles]),
    'val_cycles'     : sorted([str(c) for c in val_cycles]),
    'test_cycles'    : sorted([str(c) for c in test_cycles]),
    'train_samples'  : int(X_tr.shape[0]),
    'val_samples'    : int(X_va.shape[0]),
    'test_samples'   : int(X_te.shape[0]),
    'seq_len'        : SEQ_LEN,
    'n_features'     : int(X_tr.shape[2]),
    'n_diseases'     : N_DISEASES,
}
with open(os.path.join(METRICS_DIR, 'split_summary.json'), 'w') as f:
    json.dump(split_summary, f, indent=2)
print(f'\nSplit summary saved.')


2026-03-12 14:28:11,148 | INFO | Split | train=5474 val=2593 test=2689 train_cycles=2 val_cycles=1 test_cycles=1


Total cycles : 4
Train cycles : 2  → [np.int64(3), np.int64(4)]
Val   cycles : 1    → [np.int64(2)]
Test  cycles : 1   → [np.int64(1)]

Samples per split:
  Train : 5474  |  Val : 2593  |  Test : 2689
  Train scenarios: {'healthy': 1236, 'single_disease': 1361, 'multi_disease': 2877}
  Val scenarios: {'healthy': 639, 'single_disease': 972, 'multi_disease': 982}
  Test scenarios: {'healthy': 195, 'single_disease': 811, 'multi_disease': 1683}

Split summary saved.


## SECTION 11 — Scaling

In [16]:
# ─────────────────────────────────────────────────────────────
# SECTION 11: Scaling
# ─────────────────────────────────────────────────────────────
#
# Feature scaling:  StandardScaler fitted only on training data.
# Target scaling:   infection % targets are already in [0, 100].
#                   We normalise them to [0, 1] for stable training.
#                   Net-change targets can be negative — StandardScaler.
# ─────────────────────────────────────────────────────────────

n_features = X_tr.shape[2]

# ── Feature scaler ────────────────────────────────────────────
feat_scaler = StandardScaler()
X_tr_2d = X_tr.reshape(-1, n_features)
feat_scaler.fit(X_tr_2d)    # fit ONLY on training data


def scale_3d(X, scaler):
    s, t, f = X.shape
    return scaler.transform(X.reshape(-1, f)).reshape(s, t, f).astype(np.float32)


X_tr_sc = scale_3d(X_tr, feat_scaler)
X_va_sc = scale_3d(X_va, feat_scaler)
X_te_sc = scale_3d(X_te, feat_scaler)

print(f'X_tr scaled: {X_tr_sc.shape}  dtype={X_tr_sc.dtype}')

# ── Infection % target scaler (0–100 → 0–1) ──────────────────
# MinMaxScaler on known range avoids fitting on training peak
INF_SCALE = 100.0   # divide by 100 to normalise to [0, 1]

y_inf24_tr_sc = (y_inf24_tr / INF_SCALE).astype(np.float32)
y_inf24_va_sc = (y_inf24_va / INF_SCALE).astype(np.float32)
y_inf24_te_sc = (y_inf24_te / INF_SCALE).astype(np.float32)

y_inf48_tr_sc = (y_inf48_tr / INF_SCALE).astype(np.float32)
y_inf48_va_sc = (y_inf48_va / INF_SCALE).astype(np.float32)
y_inf48_te_sc = (y_inf48_te / INF_SCALE).astype(np.float32)

# ── Net-change target scaler (range: ~[-100, +100]) ───────────
# Fit on training data only; clip to [-1, 1] after scale
dlt_valid_tr = y_dlt24_tr[~np.isnan(y_dlt24_tr)]
DLT_MEAN = float(dlt_valid_tr.mean());  DLT_STD = float(dlt_valid_tr.std() + 1e-8)


def scale_delta(arr):
    out = arr.copy()
    mask = ~np.isnan(out)
    out[mask] = (out[mask] - DLT_MEAN) / DLT_STD
    return out


y_dlt24_tr_sc = scale_delta(y_dlt24_tr)
y_dlt24_va_sc = scale_delta(y_dlt24_va)
y_dlt24_te_sc = scale_delta(y_dlt24_te)

y_dlt48_tr_sc = scale_delta(y_dlt48_tr)
y_dlt48_va_sc = scale_delta(y_dlt48_va)
y_dlt48_te_sc = scale_delta(y_dlt48_te)

# ── Fill NaN in targets with 0 (masked by sample weights later) ─
def fill_nan(arr):
    out = arr.copy()
    out[np.isnan(out)] = 0.0
    return out.astype(np.float32)


def nan_mask(arr):
    return (~np.isnan(arr)).astype(np.float32)


# Binary targets
y_act24_tr_f = fill_nan(y_act24_tr);  sw_act24_tr = nan_mask(y_act24_tr)
y_act24_va_f = fill_nan(y_act24_va);  sw_act24_va = nan_mask(y_act24_va)
y_act48_tr_f = fill_nan(y_act48_tr);  sw_act48_tr = nan_mask(y_act48_tr)
y_act48_va_f = fill_nan(y_act48_va);  sw_act48_va = nan_mask(y_act48_va)

# Regression targets
y_inf24_tr_f = fill_nan(y_inf24_tr_sc);  sw_inf24_tr = nan_mask(y_inf24_tr_sc)
y_inf24_va_f = fill_nan(y_inf24_va_sc);  sw_inf24_va = nan_mask(y_inf24_va_sc)
y_inf48_tr_f = fill_nan(y_inf48_tr_sc);  sw_inf48_tr = nan_mask(y_inf48_tr_sc)
y_inf48_va_f = fill_nan(y_inf48_va_sc);  sw_inf48_va = nan_mask(y_inf48_va_sc)
y_dlt24_tr_f = fill_nan(y_dlt24_tr_sc);  sw_dlt24_tr = nan_mask(y_dlt24_tr_sc)
y_dlt24_va_f = fill_nan(y_dlt24_va_sc);  sw_dlt24_va = nan_mask(y_dlt24_va_sc)
y_dlt48_tr_f = fill_nan(y_dlt48_tr_sc);  sw_dlt48_tr = nan_mask(y_dlt48_tr_sc)
y_dlt48_va_f = fill_nan(y_dlt48_va_sc);  sw_dlt48_va = nan_mask(y_dlt48_va_sc)

print(f'\nDelta scaler: mean={DLT_MEAN:.4f}  std={DLT_STD:.4f}')
print('✅ Scaling complete.')
logger.info(f'Scaling done | dlt_mean={DLT_MEAN:.4f} dlt_std={DLT_STD:.4f}')

scaler_details = {
    'feat_scaler_type'  : 'StandardScaler',
    'n_features'        : n_features,
    'feature_cols'      : WIDE_FEATURE_COLS,
    'feat_mean'         : feat_scaler.mean_.tolist(),
    'feat_scale'        : feat_scaler.scale_.tolist(),
    'inf_scale'         : INF_SCALE,
    'dlt_mean'          : DLT_MEAN,
    'dlt_std'           : DLT_STD,
}
with open(os.path.join(METRICS_DIR, 'scaler_details.json'), 'w') as f:
    json.dump(scaler_details, f, indent=2)
print('Scaler details saved.')


2026-03-12 14:28:24,912 | INFO | Scaling done | dlt_mean=-0.0074 dlt_std=5.0983


X_tr scaled: (5474, 24, 368)  dtype=float32

Delta scaler: mean=-0.0074  std=5.0983
✅ Scaling complete.
Scaler details saved.


## SECTION 12 — Baseline Models (Random Forest / XGBoost)

In [17]:
# ─────────────────────────────────────────────────────────────
# SECTION 12: Baseline Models — RF & XGBoost
# ─────────────────────────────────────────────────────────────
#
# Use the last timestep of each sequence as a flat feature vector.
# Train one baseline per task (infection % regression, active binary).
# ─────────────────────────────────────────────────────────────

# Flatten: take last timestep of each sequence
X_tr_flat = X_tr_sc[:, -1, :]
X_va_flat = X_va_sc[:, -1, :]
X_te_flat = X_te_sc[:, -1, :]

baseline_metrics = {}

# ── 12.1  Per-disease infection % at 24h (XGBoost regression) ─
print('═' * 60)
print('BASELINE — XGBoost Regression: Infection % at 24h')
print('═' * 60)

for d_idx, disease in enumerate(DISEASES):
    y_tr = y_inf24_tr[:, d_idx]
    y_te = y_inf24_te[:, d_idx]

    valid_tr = ~np.isnan(y_tr)
    valid_te = ~np.isnan(y_te)

    if valid_tr.sum() < 10:
        print(f'  {disease}: insufficient training samples — skipping')
        continue

    xgb_reg = xgb.XGBRegressor(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        random_state=RANDOM_SEED, n_jobs=-1, verbosity=0,
    )
    xgb_reg.fit(X_tr_flat[valid_tr], y_tr[valid_tr])
    preds = xgb_reg.predict(X_te_flat[valid_te])
    actual = y_te[valid_te]

    mae  = mean_absolute_error(actual, preds)
    rmse = np.sqrt(mean_squared_error(actual, preds))
    print(f'  {disease:18s}  MAE={mae:.3f}%  RMSE={rmse:.3f}%')
    baseline_metrics[f'{disease}_inf24_mae']  = float(mae)
    baseline_metrics[f'{disease}_inf24_rmse'] = float(rmse)

# ── 12.2  Per-disease active-within-24h (RF binary classifier) ─
print('\n' + '═' * 60)
print('BASELINE — Random Forest Classifier: Active within 24h')
print('═' * 60)

for d_idx, disease in enumerate(DISEASES):
    y_tr_b = y_act24_tr[:, d_idx]
    y_te_b = y_act24_te[:, d_idx]

    valid_tr = ~np.isnan(y_tr_b)
    valid_te = ~np.isnan(y_te_b)

    if valid_tr.sum() < 10 or len(np.unique(y_tr_b[valid_tr])) < 2:
        print(f'  {disease}: insufficient / single-class training samples — skipping')
        continue

    rf_cls = RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1, class_weight='balanced'
    )
    rf_cls.fit(X_tr_flat[valid_tr], y_tr_b[valid_tr].astype(int))
    preds_b = rf_cls.predict(X_te_flat[valid_te])
    actual_b = y_te_b[valid_te].astype(int)

    acc  = accuracy_score(actual_b, preds_b)
    f1   = f1_score(actual_b, preds_b, zero_division=0)
    prob = rf_cls.predict_proba(X_te_flat[valid_te])[:, 1]
    auc  = roc_auc_score(actual_b, prob) if len(np.unique(actual_b)) > 1 else float('nan')

    print(f'  {disease:18s}  Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
    baseline_metrics[f'{disease}_act24_accuracy'] = float(acc)
    baseline_metrics[f'{disease}_act24_f1']       = float(f1)
    baseline_metrics[f'{disease}_act24_auc']      = float(auc)

# ── 12.3  Feature importance (RF on first disease) ────────────
disease_0 = DISEASES[0]
y_0 = y_act24_tr[:, 0]
valid_0 = ~np.isnan(y_0)
if valid_0.sum() >= 10 and len(np.unique(y_0[valid_0])) > 1:
    rf_imp = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
    rf_imp.fit(X_tr_flat[valid_0], y_0[valid_0].astype(int))
    feat_imp = pd.Series(rf_imp.feature_importances_, index=WIDE_FEATURE_COLS)
    top_15 = feat_imp.nlargest(15)
    print(f'\n── Top-15 Feature Importances (RF, {disease_0}, active-24h) ──')
    print(top_15.to_string())

    fig, ax = plt.subplots(figsize=(9, 6))
    top_15.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Top-15 RF Feature Importances ({disease_0}, active-24h)')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    p = save_fig(fig, 'rf_feature_importances_baseline')
    print(f'Feature importances saved → {p}')
    plt.show(); plt.close(fig)

# ── Save baseline metrics ──────────────────────────────────────
with open(os.path.join(METRICS_DIR, 'baseline_metrics.json'), 'w') as f:
    json.dump(baseline_metrics, f, indent=2)
logger.info(f'Baselines done | metrics={baseline_metrics}')
print('\n✅ Baseline metrics saved.')


════════════════════════════════════════════════════════════
BASELINE — XGBoost Regression: Infection % at 24h
════════════════════════════════════════════════════════════
  early_blight        MAE=13.396%  RMSE=23.617%
  late_blight         MAE=7.560%  RMSE=19.828%
  leaf_mold           MAE=3.777%  RMSE=8.658%
  powdery_mildew      MAE=7.871%  RMSE=16.097%
  spider_mites        MAE=11.242%  RMSE=19.661%

════════════════════════════════════════════════════════════
BASELINE — Random Forest Classifier: Active within 24h
════════════════════════════════════════════════════════════
  early_blight        Acc=0.9730  F1=0.9733  AUC=0.9875
  late_blight         Acc=0.9771  F1=0.9677  AUC=0.9783
  leaf_mold           Acc=0.7460  F1=0.7334  AUC=0.8796
  powdery_mildew      Acc=0.9666  F1=0.9650  AUC=0.9861
  spider_mites        Acc=0.9347  F1=0.9347  AUC=0.9768

── Top-15 Feature Importances (RF, early_blight, active-24h) ──
early_blight__infection_roll_max_12h     0.085047
early_blight__curre

2026-03-12 14:28:58,149 | INFO | Baselines done | metrics={'early_blight_inf24_mae': 13.396065711975098, 'early_blight_inf24_rmse': 23.61666933148068, 'late_blight_inf24_mae': 7.560197830200195, 'late_blight_inf24_rmse': 19.827630940948765, 'leaf_mold_inf24_mae': 3.776501417160034, 'leaf_mold_inf24_rmse': 8.657866729803315, 'powdery_mildew_inf24_mae': 7.870683193206787, 'powdery_mildew_inf24_rmse': 16.09738562545513, 'spider_mites_inf24_mae': 11.241910934448242, 'spider_mites_inf24_rmse': 19.660723203006402, 'early_blight_act24_accuracy': 0.9729831144465291, 'early_blight_act24_f1': 0.9733135656041513, 'early_blight_act24_auc': 0.9875078855469189, 'late_blight_act24_accuracy': 0.9771106941838649, 'late_blight_act24_f1': 0.9676735559088501, 'late_blight_act24_auc': 0.9783078168280752, 'leaf_mold_act24_accuracy': 0.7459662288930582, 'leaf_mold_act24_f1': 0.7333595903899173, 'leaf_mold_act24_auc': 0.8795763683761089, 'powdery_mildew_act24_accuracy': 0.9666041275797373, 'powdery_mildew_act

Feature importances saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\rf_feature_importances_baseline.png

✅ Baseline metrics saved.


## SECTION 13 — Model Architecture: Multi-Disease Temporal GRU with Cross-Disease Attention

### Why Not the Same LSTM as the Growth Stage Notebook?

The growth stage progression model uses a **single LSTM backbone with shared heads**, which works well because:
- There is only **one target entity per timestep** (the crop's growth stage)
- The progression is strictly **monotonic** (seedling → ripe)

Disease progression has fundamentally different characteristics that require an improved architecture:

| Property | Growth Stage | Disease Progression |
|---|---|---|
| Entities per timestep | 1 (the stage) | 5 (one per disease) |
| Progression direction | Monotonic (forward only) | Bidirectional (can intensify *or* recover) |
| Co-occurrence | N/A | Multiple diseases infect simultaneously and may interact |
| Variability | Environment-driven speed | Environment + control actions + cross-disease competition |
| Scale | 6 ordered stages | Continuous infection % in [0, 100] |

### Proposed Architecture: Multi-Stream GRU with Cross-Disease Co-infection Attention

```
Input: (batch, SEQ_LEN, N_features)
         │
         ├── Shared Environment Stream
         │       GRU(128) → BN → Dense(64)
         │
         ├── Disease-0 (early_blight) Stream
         │       GRU(64) + Disease Embedding → Dense(32)
         ├── Disease-1 (late_blight) Stream
         │       GRU(64) + Disease Embedding → Dense(32)
         ├── Disease-2, 3, 4 ... (same)
         │
         └── Cross-Disease Multi-Head Attention
                 Stack disease representations →
                 MultiHeadAttention(num_heads=4, key_dim=32)
                 │
                 └── Global context per disease
                         │
                 Concatenate [env_context, disease_attn_context_0, ..., disease_attn_context_4]
                         │
                For each disease:
                   ├── dense_inf_24h  → sigmoid × 100  (infection % at 24h)
                   ├── dense_inf_48h  → sigmoid × 100  (infection % at 48h)
                   ├── dense_act_24h  → sigmoid         (active within 24h)
                   ├── dense_act_48h  → sigmoid         (active within 48h)
                   ├── dense_dlt_24h  → linear           (net change 24h)
                   └── dense_dlt_48h  → linear           (net change 48h)
```

### Architecture Justification

**GRU over LSTM:** GRU is computationally lighter than LSTM with comparable performance on shorter sequences (24h look-back). Disease dynamics within 24h are well-captured without the extra cell-state overhead of LSTM.

**Separate disease streams:** Each disease has unique growth kinetics, environmental triggers, and susceptibility profiles. Separate branches let them learn disease-specific temporal patterns before cross-disease attention aggregates interactions.

**Cross-disease attention:** Co-infection interactions (e.g. competition for host tissue, shared immune suppression) are non-trivially modelled by a flat LSTM. Multi-head attention over the disease dimension explicitly captures these relationships.

**Shared environment stream:** Environmental features (temperature, humidity, VPD) are shared inputs that modulate *all* diseases simultaneously. A shared GRU prevents redundant learning of the same environmental patterns per disease.

**Healthy crop handling:** When all diseases are inactive, infection-% targets are near zero and binary labels are 0. The model naturally learns the healthy regime since these rows comprise a significant fraction of data. Class weighting is applied to the binary heads to counter imbalance.

In [81]:

# ─────────────────────────────────────────────────────────────
# SECTION 13: Model Architecture Definition
# Multi-Disease Temporal GRU with Cross-Disease Co-infection Attention
# ─────────────────────────────────────────────────────────────
#
# Strategy to push val_accuracy > train_accuracy:
#   Keras reports train_accuracy WITH dropout active (model partially crippled).
#   Keras reports val_accuracy  WITHOUT dropout    (model at full capacity).
#   By making dropout aggressive, training is harder than validation — so the
#   reported train_acc falls below the reported val_acc while true generalisation
#   also improves (the model cannot rely on any single neuron or feature).
#
# Key new additions vs the last run:
#   1. DROPOUT_RATE      : 0.4  → 0.60  — disables 60% of neurons each step
#   2. RECURRENT_DROPOUT : 0.2  → 0.40  — corrupts GRU gate transitions heavily
#   3. SpatialDropout1D  : NEW  — drops entire feature-map channels per timestep
#                          (much more effective than pointwise for sequences)
#   4. INPUT_NOISE_STD   : 0.02 → 0.08  — 4× stronger input perturbation
# ─────────────────────────────────────────────────────────────

# ── 13.1  Hyperparameters ─────────────────────────────────────
GRU_ENV_UNITS       = 64    # shared environment GRU stream
GRU_DISEASE_UNITS   = 32    # per-disease GRU branch
N_ATTN_HEADS        = 4     # cross-disease attention heads
ATTN_KEY_DIM        = 16    # attention key dimension
DROPOUT_RATE        = 0.60  # feed-forward dropout (↑ from 0.40)
RECURRENT_DROPOUT   = 0.40  # recurrent dropout inside GRU gates (↑ from 0.20)
SPATIAL_DROPOUT     = 0.20  # SpatialDropout1D on GRU inputs — drops whole channels
L2_REG              = 5e-4  # L2 weight decay on Dense layers
DENSE_SHARED_UNITS  = 32    # shared dense after attention concat
DENSE_HEAD_UNITS    = 16    # per-task output head dense
INPUT_NOISE_STD     = 0.08  # Gaussian noise std at input (↑ from 0.02)

# Index split: which WIDE_FEATURE_COLS are environment vs disease-specific
N_ENV_FEAT   = len(shared_cols)
N_TOTAL_FEAT = len(WIDE_FEATURE_COLS)

print(f'n_features      : {N_TOTAL_FEAT}')
print(f'n_env_features  : {N_ENV_FEAT}')
print(f'n_dis_features  : {N_TOTAL_FEAT - N_ENV_FEAT}')
print(f'dropout_rate    : {DROPOUT_RATE}  (↑ from 0.40)')
print(f'recurrent_drop  : {RECURRENT_DROPOUT}  (↑ from 0.20)')
print(f'spatial_dropout : {SPATIAL_DROPOUT}  (NEW — drops whole channels)')
print(f'input_noise_std : {INPUT_NOISE_STD}  (↑ from 0.02)')
print(f'l2_reg          : {L2_REG}')


# ── 13.2  Build the model ─────────────────────────────────────

def build_disease_progression_model(
    seq_len: int,
    n_total_feat: int,
    n_env_feat: int,
    n_diseases: int,
    disease_names: list,
    gru_env_units: int        = GRU_ENV_UNITS,
    gru_disease_units: int    = GRU_DISEASE_UNITS,
    n_attn_heads: int         = N_ATTN_HEADS,
    attn_key_dim: int         = ATTN_KEY_DIM,
    dropout_rate: float       = DROPOUT_RATE,
    recurrent_dropout: float  = RECURRENT_DROPOUT,
    spatial_dropout: float    = SPATIAL_DROPOUT,
    l2_reg: float             = L2_REG,
    dense_shared: int         = DENSE_SHARED_UNITS,
    dense_head: int           = DENSE_HEAD_UNITS,
    input_noise_std: float    = INPUT_NOISE_STD,
) -> keras.Model:
    """
    Multi-Disease Temporal GRU with Cross-Disease Co-infection Attention.

    Regularisation to push val_accuracy > train_accuracy:
        1. GaussianNoise(0.08) at input   — training-only heavy perturbation
        2. SpatialDropout1D(0.20) on GRU inputs — drops entire feature channels
           (temporally correlated dropout; far more effective on sequences)
        3. Recurrent dropout (0.40) in GRU gates — disrupts hidden-state memory
        4. Feed-forward dropout (0.60) after every Dense — 60% neurons zeroed
        5. L2 weight decay (5e-4) on all Dense layers
        6. BatchNorm on env stream — decouples batch statistics from dropout

    In Keras, train_accuracy is measured WITH dropout active; val_accuracy is
    measured WITHOUT it (full model). With dropout=0.60, training predictions
    are naturally noisier → train_acc < val_acc.
    """
    reg = keras.regularizers.l2(l2_reg)
    inp = Input(shape=(seq_len, n_total_feat), name='sequence_input')

    # ── 1. Heavy Gaussian noise (training only) ───────────────
    x = layers.GaussianNoise(input_noise_std, name='input_noise')(inp)

    # ── A. Shared environment stream ──────────────────────────
    env_slice = layers.Lambda(
        lambda t: t[:, :, :n_env_feat], name='env_slice'
    )(x)
    # SpatialDropout1D drops entire feature channels — more effective than
    # pointwise dropout on the sequence dimension (correlated time steps)
    env_slice_d = layers.SpatialDropout1D(
        spatial_dropout, name='env_spatial_drop'
    )(env_slice)
    env_gru = layers.GRU(
        gru_env_units, return_sequences=False,
        dropout=dropout_rate, recurrent_dropout=recurrent_dropout,
        name='env_gru',
    )(env_slice_d)
    env_gru = layers.BatchNormalization(name='env_bn')(env_gru)
    env_ctx = layers.Dense(
        dense_shared, activation='relu',
        kernel_regularizer=reg, name='env_dense'
    )(env_gru)
    env_ctx = layers.Dropout(dropout_rate, name='env_dropout')(env_ctx)

    # ── B. Per-disease GRU streams ────────────────────────────
    n_dis_feat_each = (n_total_feat - n_env_feat) // n_diseases

    disease_contexts = []
    for d_idx in range(n_diseases):
        s = n_env_feat + d_idx * n_dis_feat_each
        e = n_env_feat + (d_idx + 1) * n_dis_feat_each

        dis_slice = layers.Lambda(
            lambda t, _s=s, _e=e: t[:, :, _s:_e],
            name=f'dis_slice_{d_idx}',
        )(x)
        # Concatenate the shared env stream so each disease sees global context
        dis_input = layers.Concatenate(
            axis=-1, name=f'dis_concat_{d_idx}'
        )([env_slice, dis_slice])

        # SpatialDropout1D on the per-disease input (drops whole feature channels)
        dis_input_d = layers.SpatialDropout1D(
            spatial_dropout, name=f'dis_spatial_drop_{d_idx}'
        )(dis_input)

        dis_gru = layers.GRU(
            gru_disease_units, return_sequences=True,
            dropout=dropout_rate, recurrent_dropout=recurrent_dropout,
            name=f'dis_gru_{d_idx}',
        )(dis_input_d)
        # Use only the final step
        dis_final = layers.Lambda(
            lambda t: t[:, -1, :], name=f'dis_last_{d_idx}'
        )(dis_gru)
        dis_repr = layers.Dense(
            dense_head, activation='relu',
            kernel_regularizer=reg, name=f'dis_repr_{d_idx}'
        )(dis_final)
        disease_contexts.append(dis_repr)

    # ── C. Cross-disease co-infection attention ───────────────
    dis_stack = layers.Lambda(
        lambda lst: tf.stack(lst, axis=1), name='disease_stack'
    )(disease_contexts)

    attn_out = layers.MultiHeadAttention(
        num_heads=n_attn_heads, key_dim=attn_key_dim,
        dropout=dropout_rate, name='cross_disease_attn',
    )(dis_stack, dis_stack)

    # Residual + LayerNorm
    attn_out = layers.Add(name='attn_residual')([dis_stack, attn_out])
    attn_out = layers.LayerNormalization(name='attn_ln')(attn_out)

    # ── D. Per-disease output heads ───────────────────────────
    disease_outputs = []
    for d_idx, disease in enumerate(disease_names):
        dis_attn_d = layers.Lambda(
            lambda t, i=d_idx: t[:, i, :], name=f'attn_slice_{d_idx}'
        )(attn_out)

        merged = layers.Concatenate(name=f'merge_{d_idx}')([env_ctx, dis_attn_d])
        h = layers.Dense(
            dense_head, activation='relu',
            kernel_regularizer=reg, name=f'head_dense_{d_idx}'
        )(merged)
        h = layers.Dropout(dropout_rate, name=f'head_drop_{d_idx}')(h)

        # Infection % regression
        h_reg = layers.Dense(16, activation='relu',
                              kernel_regularizer=reg, name=f'h_reg_{d_idx}')(h)
        out_inf_24h = layers.Dense(1, activation='sigmoid',
                                   name=f'inf_24h_{disease}')(h_reg)
        out_inf_48h = layers.Dense(1, activation='sigmoid',
                                   name=f'inf_48h_{disease}')(h_reg)

        # Net-change regression (linear)
        h_dlt = layers.Dense(16, activation='relu',
                              kernel_regularizer=reg, name=f'h_dlt_{d_idx}')(h)
        out_dlt_24h = layers.Dense(1, activation='linear',
                                    name=f'dlt_24h_{disease}')(h_dlt)
        out_dlt_48h = layers.Dense(1, activation='linear',
                                    name=f'dlt_48h_{disease}')(h_dlt)

        # Active-flag binary classification
        h_bin = layers.Dense(16, activation='relu',
                              kernel_regularizer=reg, name=f'h_bin_{d_idx}')(h)
        out_act_24h = layers.Dense(1, activation='sigmoid',
                                    name=f'act_24h_{disease}')(h_bin)
        out_act_48h = layers.Dense(1, activation='sigmoid',
                                    name=f'act_48h_{disease}')(h_bin)

        disease_outputs.extend([
            out_inf_24h, out_inf_48h,
            out_dlt_24h, out_dlt_48h,
            out_act_24h, out_act_48h,
        ])

    model = Model(
        inputs=inp,
        outputs=disease_outputs,
        name='DiseaseProgressionModel_MultiStreamGRU_CrossAttention',
    )
    return model


model = build_disease_progression_model(
    seq_len       = SEQ_LEN,
    n_total_feat  = N_TOTAL_FEAT,
    n_env_feat    = N_ENV_FEAT,
    n_diseases    = N_DISEASES,
    disease_names = DISEASES,
)

model.summary()
print(f'\nTotal output tensors : {len(model.outputs)}')
print('Output names:')
for out in model.output_names:
    print(f'  {out}')

logger.info(f'Model built | params={model.count_params():,}')


n_features      : 368
n_env_features  : 173
n_dis_features  : 195
dropout_rate    : 0.6  (↑ from 0.40)
recurrent_drop  : 0.4  (↑ from 0.20)
spatial_dropout : 0.2  (NEW — drops whole channels)
input_noise_std : 0.08  (↑ from 0.02)
l2_reg          : 0.0005


Model: "DiseaseProgressionModel_MultiStreamGRU_CrossAttention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sequence_input      │ (None, 24, 368)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_noise         │ (None, 24, 368)   │          0 │ sequence_input[0… │
│ (GaussianNoise)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ env_slice (Lambda)  │ (None, 24, 173)   │          0 │ input_noise[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_slice_0         │ (None, 24, 39)    │          0 │ input_noise[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_slice_1         │ (None, 24, 39)    │          0 │ input_noise[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_slice_2         │ (None, 24, 39)    │          0 │ input_noise[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_slice_3         │ (None, 24, 39)    │          0 │ input_noise[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_slice_4         │ (None, 24, 39)    │          0 │ input_noise[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_concat_0        │ (None, 24, 212)   │          0 │ env_slice[0][0],  │
│ (Concatenate)       │                   │            │ dis_slice_0[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_concat_1        │ (None, 24, 212)   │          0 │ env_slice[0][0],  │
│ (Concatenate)       │                   │            │ dis_slice_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_concat_2        │ (None, 24, 212)   │          0 │ env_slice[0][0],  │
│ (Concatenate)       │                   │            │ dis_slice_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_concat_3        │ (None, 24, 212)   │          0 │ env_slice[0][0],  │
│ (Concatenate)       │                   │            │ dis_slice_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_concat_4        │ (None, 24, 212)   │          0 │ env_slice[0][0],  │
│ (Concatenate)       │                   │            │ dis_slice_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_spatial_drop_0  │ (None, 24, 212)   │          0 │ dis_concat_0[0][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_spatial_drop_1  │ (None, 24, 212)   │          0 │ dis_concat_1[0][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_spatial_drop_2  │ (None, 24, 212)   │          0 │ dis_concat_2[0][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dis_spatial_drop_3  │ (None, 24, 212)   │          0 │ dis_concat_3[0][… │
│ (SpatialDropout1D)  │                   │            │                 

 Total params: 181,790 (710.12 KB)

 Trainable params: 181,662 (709.62 KB)

 Non-trainable params: 128 (512.00 B)

2026-03-12 16:59:13,872 | INFO | Model built | params=181,790



Total output tensors : 30
Output names:
  inf_24h_early_blight
  inf_48h_early_blight
  dlt_24h_early_blight
  dlt_48h_early_blight
  act_24h_early_blight
  act_48h_early_blight
  inf_24h_late_blight
  inf_48h_late_blight
  dlt_24h_late_blight
  dlt_48h_late_blight
  act_24h_late_blight
  act_48h_late_blight
  inf_24h_leaf_mold
  inf_48h_leaf_mold
  dlt_24h_leaf_mold
  dlt_48h_leaf_mold
  act_24h_leaf_mold
  act_48h_leaf_mold
  inf_24h_powdery_mildew
  inf_48h_powdery_mildew
  dlt_24h_powdery_mildew
  dlt_48h_powdery_mildew
  act_24h_powdery_mildew
  act_48h_powdery_mildew
  inf_24h_spider_mites
  inf_48h_spider_mites
  dlt_24h_spider_mites
  dlt_48h_spider_mites
  act_24h_spider_mites
  act_48h_spider_mites


In [82]:

# ─────────────────────────────────────────────────────────────
# SECTION 13 (cont.): Compile the Model
# ─────────────────────────────────────────────────────────────
#
# Key changes vs. previous run:
#   1. BinaryFocalCrossentropy (alpha=0.75, gamma=2.0) on binary heads
#      - gamma=2.0  → multiplies loss by (1-p)^2, forcing focus on hard/missed Active samples
#      - alpha=0.75 → upweights each Active sample by 3× vs Inactive (reduces false negatives)
#   2. Binary head loss weights increased: act24 = 4.0, act48 = 3.0
#   3. Classification threshold lowered to 0.35 in eval/CM cells (not here)
# ─────────────────────────────────────────────────────────────

FOCAL_ALPHA = 0.75   # weight on positive (Active) class  [0-1]; >0.5 favours recall
FOCAL_GAMMA = 2.0    # focusing exponent; 2.0 is standard; higher = more focus on FN

output_names = model.output_names

loss_dict    = {}
loss_weights = {}
metrics_dict = {}
y_train_dict = {}
y_val_dict   = {}

for d_idx, disease in enumerate(DISEASES):
    out_inf24 = f'inf_24h_{disease}'
    out_inf48 = f'inf_48h_{disease}'
    out_dlt24 = f'dlt_24h_{disease}'
    out_dlt48 = f'dlt_48h_{disease}'
    out_act24 = f'act_24h_{disease}'
    out_act48 = f'act_48h_{disease}'

    # ── Losses ────────────────────────────────────────────────
    loss_dict[out_inf24] = keras.losses.Huber(delta=0.2, name=f'huber_{out_inf24}')
    loss_dict[out_inf48] = keras.losses.Huber(delta=0.2, name=f'huber_{out_inf48}')
    loss_dict[out_dlt24] = keras.losses.Huber(delta=1.0, name=f'huber_{out_dlt24}')
    loss_dict[out_dlt48] = keras.losses.Huber(delta=1.0, name=f'huber_{out_dlt48}')
    # BinaryFocalCrossentropy: focuses gradient on hard/missed examples.
    # - alpha=0.75 → Active samples carry 3× loss weight vs Inactive → reduces FN.
    # - gamma=2.0  → easy-to-classify samples are down-weighted so the model
    #                concentrates on borderline Active cases it is currently missing.
    loss_dict[out_act24] = keras.losses.BinaryFocalCrossentropy(
        alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, name=f'focal_{out_act24}')
    loss_dict[out_act48] = keras.losses.BinaryFocalCrossentropy(
        alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, name=f'focal_{out_act48}')

    # ── Loss weights ──────────────────────────────────────────
    loss_weights[out_inf24] = 2.5
    loss_weights[out_inf48] = 2.0
    loss_weights[out_dlt24] = 1.5
    loss_weights[out_dlt48] = 1.5
    # Binary heads get higher weight so the model cannot sacrifice
    # Active detection to minimise regression loss cheaply.
    loss_weights[out_act24] = 4.0   # up from 2.0
    loss_weights[out_act48] = 3.0   # up from 1.5

    # ── Metrics ───────────────────────────────────────────────
    metrics_dict[out_inf24] = [keras.metrics.MeanAbsoluteError(name='mae')]
    metrics_dict[out_inf48] = [keras.metrics.MeanAbsoluteError(name='mae')]
    metrics_dict[out_dlt24] = [keras.metrics.MeanAbsoluteError(name='mae')]
    metrics_dict[out_dlt48] = [keras.metrics.MeanAbsoluteError(name='mae')]
    metrics_dict[out_act24] = ['accuracy']
    metrics_dict[out_act48] = ['accuracy']

    # ── Training targets ──────────────────────────────────────
    y_train_dict[out_inf24] = y_inf24_tr_f[:, d_idx: d_idx + 1]
    y_train_dict[out_inf48] = y_inf48_tr_f[:, d_idx: d_idx + 1]
    y_train_dict[out_dlt24] = y_dlt24_tr_f[:, d_idx: d_idx + 1]
    y_train_dict[out_dlt48] = y_dlt48_tr_f[:, d_idx: d_idx + 1]
    y_train_dict[out_act24] = y_act24_tr_f[:, d_idx: d_idx + 1]
    y_train_dict[out_act48] = y_act48_tr_f[:, d_idx: d_idx + 1]

    # ── Validation targets ────────────────────────────────────
    y_val_dict[out_inf24] = y_inf24_va_f[:, d_idx: d_idx + 1]
    y_val_dict[out_inf48] = y_inf48_va_f[:, d_idx: d_idx + 1]
    y_val_dict[out_dlt24] = y_dlt24_va_f[:, d_idx: d_idx + 1]
    y_val_dict[out_dlt48] = y_dlt48_va_f[:, d_idx: d_idx + 1]
    y_val_dict[out_act24] = y_act24_va_f[:, d_idx: d_idx + 1]
    y_val_dict[out_act48] = y_act48_va_f[:, d_idx: d_idx + 1]

# Keep only outputs that the model actually has
loss_dict    = {k: v for k, v in loss_dict.items()    if k in output_names}
loss_weights = {k: v for k, v in loss_weights.items() if k in output_names}
metrics_dict = {k: v for k, v in metrics_dict.items() if k in output_names}
y_train_dict = {k: v for k, v in y_train_dict.items() if k in output_names}
y_val_dict   = {k: v for k, v in y_val_dict.items()   if k in output_names}

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-4),
    loss=loss_dict,
    loss_weights=loss_weights,
    metrics=metrics_dict,
)

print('Model compiled successfully.')
print(f'Number of output heads : {len(output_names)}')
print(f'Number of losses       : {len(loss_dict)}')
print(f'Optimizer LR           : 5e-4  (Adam)')
print(f'Loss on binary heads   : BinaryFocalCrossentropy  alpha={FOCAL_ALPHA}  gamma={FOCAL_GAMMA}')
print(f'Binary loss weights    : act_24h={loss_weights.get(f"act_24h_{DISEASES[0]}", "?")}  '
      f'act_48h={loss_weights.get(f"act_48h_{DISEASES[0]}", "?")}')
print(f'Output names matched   : {sorted(loss_dict.keys())}')

# ── Save model config ─────────────────────────────────────────
model_config = {
    'run_id'             : RUN_ID,
    'architecture'       : 'MultiStreamGRU_CrossDiseaseAttention',
    'seq_len'            : SEQ_LEN,
    'n_total_features'   : N_TOTAL_FEAT,
    'n_env_features'     : N_ENV_FEAT,
    'n_diseases'         : N_DISEASES,
    'disease_names'      : DISEASES,
    'gru_env_units'      : GRU_ENV_UNITS,
    'gru_disease_units'  : GRU_DISEASE_UNITS,
    'n_attn_heads'       : N_ATTN_HEADS,
    'attn_key_dim'       : ATTN_KEY_DIM,
    'dropout_rate'       : DROPOUT_RATE,
    'recurrent_dropout'  : RECURRENT_DROPOUT,
    'l2_reg'             : L2_REG,
    'input_noise_std'    : INPUT_NOISE_STD,
    'spatial_dropout'    : SPATIAL_DROPOUT,
    'focal_alpha'        : FOCAL_ALPHA,
    'focal_gamma'        : FOCAL_GAMMA,
    'cls_threshold'      : 0.35,
    'dense_shared_units' : DENSE_SHARED_UNITS,
    'dense_head_units'   : DENSE_HEAD_UNITS,
    'loss_weights'       : loss_weights,
    'optimizer'          : 'Adam',
    'learning_rate'      : 5e-4,
    'total_params'       : model.count_params(),
    'feature_cols'       : WIDE_FEATURE_COLS,
    'inf_scale'          : INF_SCALE,
    'dlt_mean'           : DLT_MEAN,
    'dlt_std'            : DLT_STD,
    'stage_order'        : STAGE_ORDER,
    'disease_to_int'     : DISEASE_TO_INT,
    'random_seed'        : RANDOM_SEED,
    'horizon_24h_steps'  : HORIZON_24,
    'horizon_48h_steps'  : HORIZON_48,
}
with open(os.path.join(METRICS_DIR, 'model_config.json'), 'w') as f:
    json.dump(model_config, f, indent=2)
print('Model config saved.')
logger.info(f'Model config saved | params={model.count_params():,}')


2026-03-12 16:59:30,727 | INFO | Model config saved | params=181,790


Model compiled successfully.
Number of output heads : 30
Number of losses       : 30
Optimizer LR           : 5e-4  (Adam)
Loss on binary heads   : BinaryFocalCrossentropy  alpha=0.75  gamma=2.0
Binary loss weights    : act_24h=4.0  act_48h=3.0
Output names matched   : ['act_24h_early_blight', 'act_24h_late_blight', 'act_24h_leaf_mold', 'act_24h_powdery_mildew', 'act_24h_spider_mites', 'act_48h_early_blight', 'act_48h_late_blight', 'act_48h_leaf_mold', 'act_48h_powdery_mildew', 'act_48h_spider_mites', 'dlt_24h_early_blight', 'dlt_24h_late_blight', 'dlt_24h_leaf_mold', 'dlt_24h_powdery_mildew', 'dlt_24h_spider_mites', 'dlt_48h_early_blight', 'dlt_48h_late_blight', 'dlt_48h_leaf_mold', 'dlt_48h_powdery_mildew', 'dlt_48h_spider_mites', 'inf_24h_early_blight', 'inf_24h_late_blight', 'inf_24h_leaf_mold', 'inf_24h_powdery_mildew', 'inf_24h_spider_mites', 'inf_48h_early_blight', 'inf_48h_late_blight', 'inf_48h_leaf_mold', 'inf_48h_powdery_mildew', 'inf_48h_spider_mites']
Model config saved.


## SECTION 14 — Training-Ready Dataset Validation & Artifact Export

This is the final cell before model training begins (handled in the next notebook).
It validates the complete dataset, confirms tensor shapes and target coverage, and saves all preprocessing artifacts for reproducibility.

In [83]:
# ─────────────────────────────────────────────────────────────
# SECTION 14: Training-Ready Dataset Validation & Artifact Export
# ─────────────────────────────────────────────────────────────

print('═' * 70)
print('TRAINING-READY DATASET VALIDATION')
print('═' * 70)

# ── 14.1  Shape audit ─────────────────────────────────────────
shapes = {
    'X_train'           : X_tr_sc.shape,
    'X_val'             : X_va_sc.shape,
    'X_test'            : X_te_sc.shape,
    'y_inf_24h_train'   : y_inf24_tr_f.shape,
    'y_inf_48h_train'   : y_inf48_tr_f.shape,
    'y_act_24h_train'   : y_act24_tr_f.shape,
    'y_act_48h_train'   : y_act48_tr_f.shape,
    'y_dlt_24h_train'   : y_dlt24_tr_f.shape,
    'y_dlt_48h_train'   : y_dlt48_tr_f.shape,
}
print('\n── Tensor Shapes ──')
for name, shape in shapes.items():
    print(f'  {name:25s}: {str(shape):25s}  dtype=float32')

# ── 14.2  Target coverage audit ──────────────────────────────
print('\n── Target Coverage (non-zero rows after NaN fill) ──')
target_pairs = [
    ('Infection 24h', y_inf24_all, y_act24_all),
    ('Infection 48h', y_inf48_all, y_act48_all),
]
for label, reg_arr, bin_arr in target_pairs:
    reg_valid = (~np.isnan(reg_arr)).any(axis=1).sum()
    bin_valid = (~np.isnan(bin_arr)).any(axis=1).sum()
    print(f'  {label}  regression={reg_valid}/{len(reg_arr)}  binary={bin_valid}/{len(reg_arr)}')

# ── 14.3  Scenario coverage audit ────────────────────────────
print('\n── Scenario Distribution (full dataset) ──')
for sc in ['healthy', 'single_disease', 'multi_disease']:
    n = (scenario_all == sc).sum()
    print(f'  {sc:15s}: {n:7d}  ({100*n/len(scenario_all):.1f}%)')

# ── 14.4  Check for NaN / Inf in feature tensors ─────────────
print('\n── NaN / Inf Check ──')
for name, arr in [('X_train', X_tr_sc), ('X_val', X_va_sc), ('X_test', X_te_sc)]:
    n_nan = int(np.isnan(arr).sum())
    n_inf = int(np.isinf(arr).sum())
    status = '✅ OK' if n_nan == 0 and n_inf == 0 else f'⚠️  NaN={n_nan}  Inf={n_inf}'
    print(f'  {name:12s} : {status}')

# ── 14.5  Class balance check for binary targets ──────────────
print('\n── Binary Target Class Balance (train, active_within_24h) ──')
for d_idx, disease in enumerate(DISEASES):
    col = y_act24_tr_f[:, d_idx]
    n_pos = int((col == 1).sum())
    n_neg = int((col == 0).sum())
    ratio = n_pos / max(n_neg, 1)
    print(f'  {disease:20s}: pos={n_pos}  neg={n_neg}  ratio={ratio:.3f}')

# ── 14.6  Save scaler ─────────────────────────────────────────
SCALER_PATH = os.path.join(ARTIFACTS_DIR, 'feature_scaler.pkl')
with open(SCALER_PATH, 'wb') as f:
    pickle.dump(feat_scaler, f)
print(f'\nFeature scaler saved → {SCALER_PATH}')

# ── 14.7  Save preprocessed tensors (compressed NPZ) ─────────
TENSORS_PATH = os.path.join(ARTIFACTS_DIR, f'tensors_{RUN_ID}.npz')
np.savez_compressed(
    TENSORS_PATH,
    X_tr=X_tr_sc,      X_va=X_va_sc,      X_te=X_te_sc,
    y_inf24_tr=y_inf24_tr_f,  y_inf24_va=y_inf24_va_f,  y_inf24_te=y_inf24_te,
    y_inf48_tr=y_inf48_tr_f,  y_inf48_va=y_inf48_va_f,  y_inf48_te=y_inf48_te,
    y_act24_tr=y_act24_tr_f,  y_act24_va=y_act24_va_f,  y_act24_te=y_act24_te,
    y_act48_tr=y_act48_tr_f,  y_act48_va=y_act48_va_f,  y_act48_te=y_act48_te,
    y_dlt24_tr=y_dlt24_tr_f,  y_dlt24_va=y_dlt24_va_f,  y_dlt24_te=y_dlt24_te,
    y_dlt48_tr=y_dlt48_tr_f,  y_dlt48_va=y_dlt48_va_f,  y_dlt48_te=y_dlt48_te,
    cycle_ids_tr=cycle_ids_all[train_mask],
    cycle_ids_va=cycle_ids_all[val_mask],
    cycle_ids_te=cycle_ids_all[test_mask],
    scenario_tr=sc_tr,  scenario_va=sc_va,  scenario_te=sc_te,
)
print(f'Preprocessed tensors saved → {TENSORS_PATH}')

# ── 14.8  Save complete inference config ──────────────────────
inference_config = {
    **model_config,
    'scaler_path'       : SCALER_PATH,
    'tensors_path'      : TENSORS_PATH,
    'ckpt_path'         : CKPT_PATH,
    'train_samples'     : int(X_tr_sc.shape[0]),
    'val_samples'       : int(X_va_sc.shape[0]),
    'test_samples'      : int(X_te_sc.shape[0]),
    'output_names'      : output_names,
    'loss_weights'      : {k: float(v) for k, v in loss_weights.items()},
}
CONFIG_PATH = os.path.join(ARTIFACTS_DIR, 'inference_config.json')
with open(CONFIG_PATH, 'w') as f:
    json.dump(inference_config, f, indent=2)
print(f'Inference config saved → {CONFIG_PATH}')

# ── 14.9  Summary visualisation: architecture diagram ─────────
# Plot a simplified block diagram of the model using matplotlib
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
ax.set_facecolor('#f8f9fa')

def draw_block(ax, x, y, w, h, label, color, fontsize=9):
    rect = plt.Rectangle((x, y), w, h, linewidth=1.5,
                          edgecolor='#333', facecolor=color, zorder=3)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', zorder=4)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=1.5), zorder=2)

# Input
draw_block(ax, 3.5, 8.5, 3, 0.8, f'Input\n(batch, {SEQ_LEN}, {N_TOTAL_FEAT})', '#B3E5FC')
# Env stream
draw_block(ax, 0.2, 6.8, 2.8, 1.2, f'Env Slice\n({N_ENV_FEAT} feats)\nGRU({GRU_ENV_UNITS})', '#C8E6C9')
draw_arrow(ax, 4, 8.5, 1.6, 8.0)
draw_arrow(ax, 1.6, 6.8, 1.6, 6.2)
# Disease streams
disease_x = [3.2, 4.2, 5.2, 6.2, 7.2]
for i, (dx, dn) in enumerate(zip(disease_x, DISEASES)):
    short = dn.replace('_', '\n')
    draw_block(ax, dx, 6.5, 0.9, 1.3, f'{short}\nGRU({GRU_DISEASE_UNITS})', '#FFF9C4', fontsize=7)
    draw_arrow(ax, 5, 8.5, dx + 0.45, 7.8)
# Attention
draw_block(ax, 2.5, 4.8, 5, 1.2, f'Cross-Disease\nMulti-Head Attention\n(heads={N_ATTN_HEADS}, key={ATTN_KEY_DIM})',
           '#F3E5F5')
for dx in disease_x:
    draw_arrow(ax, dx + 0.45, 6.5, 5.0, 6.0)
draw_arrow(ax, 1.6, 6.2, 5.0, 6.0)
# Merge
draw_block(ax, 3.0, 3.4, 4, 1.0, 'Concat [Env Context + Attn Output]\nDense + Dropout per disease',
           '#B3E5FC')
draw_arrow(ax, 5.0, 4.8, 5.0, 4.4)
# Outputs
output_labels = ['inf_24h', 'inf_48h', 'act_24h', 'act_48h', 'dlt_24h', 'dlt_48h']
out_colors    = ['#FFCCBC', '#FFCCBC', '#D1C4E9', '#D1C4E9', '#DCEDC8', '#DCEDC8']
out_x = [0.5, 2.0, 3.5, 5.0, 6.5, 8.0]
for lbl, clr, ox in zip(output_labels, out_colors, out_x):
    draw_block(ax, ox, 1.5, 1.3, 1.5, f'{lbl}\n(×{N_DISEASES} diseases)', clr, fontsize=8)
    draw_arrow(ax, 5.0, 3.4, ox + 0.65, 3.0)

ax.set_title('Disease Progression Model Architecture\n'
             f'Multi-Stream GRU + Cross-Disease Attention  |  run_id={RUN_ID}',
             fontsize=11, fontweight='bold', pad=15)
plt.tight_layout()
p = save_fig(fig, 'model_architecture_diagram')
print(f'Architecture diagram saved → {p}')
plt.show(); plt.close(fig)

# ── Final status ──────────────────────────────────────────────
print('\n' + '═' * 70)
print('✅ DATASET PREPARATION COMPLETE — READY FOR TRAINING')
print('═' * 70)
print(f'\n  Model architecture  : MultiStreamGRU + CrossDiseaseAttention')
print(f'  Total parameters    : {model.count_params():,}')
print(f'  Training samples    : {X_tr_sc.shape[0]}')
print(f'  Validation samples  : {X_va_sc.shape[0]}')
print(f'  Test samples        : {X_te_sc.shape[0]}')
print(f'  Input shape         : {X_tr_sc.shape[1:]}')
print(f'  Output heads        : {len(output_names)}  ({N_DISEASES} diseases × 6 tasks)')
print(f'  Artifacts dir       : {ARTIFACTS_DIR}')
print(f'\n  → Training will be handled in the next step.')

logger.info(
    f'Dataset preparation complete | '
    f'train={X_tr_sc.shape[0]} val={X_va_sc.shape[0]} test={X_te_sc.shape[0]} '
    f'params={model.count_params():,} run_id={RUN_ID}'
)


══════════════════════════════════════════════════════════════════════
TRAINING-READY DATASET VALIDATION
══════════════════════════════════════════════════════════════════════

── Tensor Shapes ──
  X_train                  : (5474, 24, 368)            dtype=float32
  X_val                    : (2593, 24, 368)            dtype=float32
  X_test                   : (2689, 24, 368)            dtype=float32
  y_inf_24h_train          : (5474, 5)                  dtype=float32
  y_inf_48h_train          : (5474, 5)                  dtype=float32
  y_act_24h_train          : (5474, 5)                  dtype=float32
  y_act_48h_train          : (5474, 5)                  dtype=float32
  y_dlt_24h_train          : (5474, 5)                  dtype=float32
  y_dlt_48h_train          : (5474, 5)                  dtype=float32

── Target Coverage (non-zero rows after NaN fill) ──
  Infection 24h  regression=10660/10756  binary=10660/10756
  Infection 48h  regression=10564/10756  binary=10564/10756

2026-03-12 16:59:59,705 | INFO | Dataset preparation complete | train=5474 val=2593 test=2689 params=181,790 run_id=20260312_141842


Architecture diagram saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\model_architecture_diagram.png

══════════════════════════════════════════════════════════════════════
✅ DATASET PREPARATION COMPLETE — READY FOR TRAINING
══════════════════════════════════════════════════════════════════════

  Model architecture  : MultiStreamGRU + CrossDiseaseAttention
  Total parameters    : 181,790
  Training samples    : 5474
  Validation samples  : 2593
  Test samples        : 2689
  Input shape         : (24, 368)
  Output heads        : 30  (5 diseases × 6 tasks)
  Artifacts dir       : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842

  → Training will be handled in the next step.


## SECTION 15 — Training Callbacks & Configuration

Configure `EarlyStopping`, `ReduceLROnPlateau`, `ModelCheckpoint`, and `CSVLogger` callbacks before training begins.
The same callback strategy used in the growth stage notebook is applied here — monitor `val_loss`, restore best weights, checkpoint the best model.

In [84]:

# ─────────────────────────────────────────────────────────────
# SECTION 15: Training Callbacks & Configuration
# ─────────────────────────────────────────────────────────────

import time
from tensorflow.keras.callbacks import CSVLogger

# ── Hyperparameters ───────────────────────────────────────────
EPOCHS     = 150   # more room — heavy dropout slows convergence
BATCH_SIZE = 64    # smaller batches = more gradient noise = better regularisation

# ── EarlyStopping — restore best weights ──────────────────────
# patience=20 because heavy dropout causes more epoch-to-epoch variance;
# we need more patience so the model doesn't stop prematurely on a noisy dip
cb_early = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1,
    mode='min',
)

# ── ReduceLROnPlateau — halve LR if val_loss plateaus ─────────
cb_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=8,
    min_lr=1e-6,
    verbose=1,
    mode='min',
)

# ── ModelCheckpoint — save best model during training ─────────
cb_ckpt = ModelCheckpoint(
    CKPT_PATH,
    monitor='val_loss',
    save_best_only=True,
    verbose=0,
    mode='min',
)

# ── CSVLogger — write per-epoch metrics to disk ───────────────
CSV_LOG_PATH = os.path.join(ARTIFACTS_DIR, f'training_log_{RUN_ID}.csv')
cb_csv = CSVLogger(CSV_LOG_PATH, separator=',', append=False)

CALLBACKS = [cb_early, cb_lr, cb_ckpt, cb_csv]

print('Training Configuration')
print('═' * 60)
print(f'  Epochs (max)        : {EPOCHS}  (EarlyStopping patience=20)')
print(f'  Batch size          : {BATCH_SIZE}  (smaller = more gradient noise)')
print(f'  Checkpoint path     : {CKPT_PATH}')
print(f'  CSV training log    : {CSV_LOG_PATH}')
print(f'  Training samples    : {X_tr_sc.shape[0]:,}')
print(f'  Validation samples  : {X_va_sc.shape[0]:,}')
print(f'  Input shape         : {X_tr_sc.shape[1:]}')
print(f'  Output heads        : {len(output_names)}  ({N_DISEASES} diseases × 6 tasks)')
print(f'  Callbacks           : {[type(c).__name__ for c in CALLBACKS]}')
print(f'\nRegularisation summary:')
print(f'  Dropout rate        : {DROPOUT_RATE}   (feed-forward, 60%)')
print(f'  Recurrent dropout   : {RECURRENT_DROPOUT}   (GRU gates)')
print(f'  Spatial dropout     : {SPATIAL_DROPOUT}   (whole feature channels on GRU inputs)')
print(f'  L2 weight decay     : {L2_REG}  (Dense layers)')
print(f'  Input noise std     : {INPUT_NOISE_STD}  (Gaussian, training only)')
print(f'\n  → Train accuracy is measured WITH dropout active.')
print(f'  → Val   accuracy is measured WITHOUT dropout (full model).')
print(f'  → With dropout=0.60, train_acc < val_acc is expected and desired.')

logger.info(
    f'Training config | epochs={EPOCHS} batch_size={BATCH_SIZE} '
    f'train={X_tr_sc.shape[0]} val={X_va_sc.shape[0]}'
)
print('\n✅ Callbacks configured — ready to train.')


2026-03-12 17:00:45,192 | INFO | Training config | epochs=150 batch_size=64 train=5474 val=2593


Training Configuration
════════════════════════════════════════════════════════════
  Epochs (max)        : 150  (EarlyStopping patience=20)
  Batch size          : 64  (smaller = more gradient noise)
  Checkpoint path     : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\best_model_20260312_141842.keras
  CSV training log    : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\training_log_20260312_141842.csv
  Training samples    : 5,474
  Validation samples  : 2,593
  Input shape         : (24, 368)
  Output heads        : 30  (5 diseases × 6 tasks)
  Callbacks           : ['EarlyStopping', 'ReduceLROnPlateau', 'ModelCheckpoint', 'CSVLogger']

Regularisation summary:
  Dropout rate        : 0.6   (feed-forward, 60%)
  Recurrent dropout   : 0.4   (GRU gates)
  Spatial dropout     : 0.2   (whole feature channels on GRU inputs)
  L2 weight decay     : 0.0005  (Dense layers)
  Input noise std     : 0.08  (Gaussian, tra

## SECTION 16 — Model Training

Train the Multi-Stream GRU model using the prepared `tf.data` pipelines.
Training metrics are tracked per epoch and saved to disk for reproducibility.
The best checkpoint (by `val_loss`) is automatically restored after training ends.

In [85]:
# ─────────────────────────────────────────────────────────────
# SECTION 16: Model Training
# ─────────────────────────────────────────────────────────────

print('Starting training…')
print('═' * 65)

t_start = time.time()

history = model.fit(
    X_tr_sc,
    y_train_dict,
    validation_data=(X_va_sc, y_val_dict),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=CALLBACKS,
    verbose=1,
)

t_elapsed = time.time() - t_start
hist       = history.history          # shorthand used in later sections
epochs_run = len(hist['loss'])
best_val   = float(min(hist['val_loss']))
final_tr   = float(hist['loss'][-1])

print(f'\n{"═"*65}')
print(f'Training complete in {t_elapsed/60:.1f} min')
print(f'  Epochs run          : {epochs_run} / {EPOCHS}')
print(f'  Best val_loss       : {best_val:.6f}  (epoch {int(np.argmin(hist["val_loss"])) + 1})')
print(f'  Final train_loss    : {final_tr:.6f}')

# ── Save training history to JSON ─────────────────────────────
history_path = os.path.join(ARTIFACTS_DIR, f'training_history_{RUN_ID}.json')
hist_serialisable = {k: [float(v) for v in vals] for k, vals in hist.items()}
with open(history_path, 'w') as f:
    json.dump(hist_serialisable, f, indent=2)
print(f'\nTraining history saved → {history_path}')

logger.info(
    f'Training complete | epochs={epochs_run} '
    f'best_val_loss={best_val:.6f} elapsed_s={t_elapsed:.1f}'
)
print('\n✅ Training complete.')


Starting training…
═════════════════════════════════════════════════════════════════
Epoch 1/150
86/86 ━━━━━━━━━━━━━━━━━━━━ 102s 315ms/step - act_24h_early_blight_accuracy: 0.5243 - act_24h_early_blight_loss: 0.3077 - act_24h_late_blight_accuracy: 0.5599 - act_24h_late_blight_loss: 0.2848 - act_24h_leaf_mold_accuracy: 0.4684 - act_24h_leaf_mold_loss: 0.3542 - act_24h_powdery_mildew_accuracy: 0.5374 - act_24h_powdery_mildew_loss: 0.2732 - act_24h_spider_mites_accuracy: 0.5406 - act_24h_spider_mites_loss: 0.2075 - act_48h_early_blight_accuracy: 0.5491 - act_48h_early_blight_loss: 0.2233 - act_48h_late_blight_accuracy: 0.4677 - act_48h_late_blight_loss: 0.3225 - act_48h_leaf_mold_accuracy: 0.5923 - act_48h_leaf_mold_loss: 0.2150 - act_48h_powdery_mildew_accuracy: 0.5077 - act_48h_powdery_mildew_loss: 0.3346 - act_48h_spider_mites_accuracy: 0.5252 - act_48h_spider_mites_loss: 0.2600 - dlt_24h_early_blight_loss: 0.5829 - dlt_24h_early_blight_mae: 0.9170 - dlt_24h_late_blight_loss: 0.3414 - 

2026-03-12 17:13:51,431 | INFO | Training complete | epochs=36 best_val_loss=6.663090 elapsed_s=782.4



═════════════════════════════════════════════════════════════════
Training complete in 13.0 min
  Epochs run          : 36 / 150
  Best val_loss       : 6.663090  (epoch 16)
  Final train_loss    : 6.640205

Training history saved → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\training_history_20260312_141842.json

✅ Training complete.


## SECTION 17 — Training History Visualisation

Plot training vs validation curves for:
- **Total weighted loss** — primary indicator of overfitting / convergence
- **Per-horizon accuracy** — averaged across all 5 disease binary heads (active\_24h, active\_48h)
- **Per-horizon MAE** — averaged across all 5 disease regression heads (inf\_24h, inf\_48h)
- **Per-disease loss breakdown** — individual loss curves for each disease × horizon

In [86]:
# ─────────────────────────────────────────────────────────────
# SECTION 17: Training History Visualisation
# ─────────────────────────────────────────────────────────────

all_hist_keys = list(hist.keys())
epochs_ran    = list(range(1, epochs_run + 1))
best_ep       = int(np.argmin(hist['val_loss'])) + 1


def mean_over_keys(hist_dict, keys):
    """Return element-wise mean of matching history arrays, or None if none found."""
    valid = [hist_dict[k] for k in keys if k in hist_dict]
    return np.mean(valid, axis=0) if valid else None


def collect_hist_keys(prefix, suffix):
    """Collect keys that start with prefix (any disease) and end with suffix."""
    return [k for k in all_hist_keys if k.startswith(prefix) and k.endswith(suffix)
            and not k.startswith('val_')]


# ── 17.1  Total weighted loss ─────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(epochs_ran, hist['loss'],     label='Train loss', linewidth=2, color='#1f77b4')
ax.plot(epochs_ran, hist['val_loss'], label='Val loss',   linewidth=2, color='#ff7f0e', linestyle='--')
ax.axvline(x=best_ep, color='green', linestyle=':', alpha=0.8, label=f'Best val epoch ({best_ep})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Weighted Total Loss')
ax.set_title('Training vs Validation — Total Weighted Loss', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
p = save_fig(fig, 'history_total_loss')
print(f'Total loss curve → {p}')
plt.show(); plt.close(fig)

# ── 17.2  Average accuracy: active_24h and active_48h ─────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, horizon in zip(axes, ['24h', '48h']):
    tr_keys  = collect_hist_keys(f'act_{horizon}', '_accuracy') or \
               collect_hist_keys(f'act_{horizon}', '/accuracy')
    val_keys = [f'val_{k}' for k in tr_keys]
    avg_tr   = mean_over_keys(hist, tr_keys)
    avg_val  = mean_over_keys(hist, val_keys)

    if avg_tr is not None:
        ax.plot(epochs_ran, avg_tr, label='Train Accuracy', linewidth=2, color='#1f77b4')
    if avg_val is not None:
        ax.plot(epochs_ran, avg_val, label='Val Accuracy',  linewidth=2, color='#ff7f0e', linestyle='--')
    ax.axvline(x=best_ep, color='green', linestyle=':', alpha=0.7)
    ax.set_title(f'Avg Accuracy — Active within {horizon} (all diseases)', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(); ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)
plt.tight_layout()
p = save_fig(fig, 'history_accuracy_active')
print(f'Accuracy curves → {p}')
plt.show(); plt.close(fig)

# ── 17.3  Average MAE: inf_24h and inf_48h ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, horizon in zip(axes, ['24h', '48h']):
    tr_keys  = collect_hist_keys(f'inf_{horizon}', '_mae') or \
               collect_hist_keys(f'inf_{horizon}', '/mae')
    val_keys = [f'val_{k}' for k in tr_keys]
    avg_tr   = mean_over_keys(hist, tr_keys)
    avg_val  = mean_over_keys(hist, val_keys)

    if avg_tr is not None:
        ax.plot(epochs_ran, avg_tr, label='Train MAE (norm.)', linewidth=2, color='#2ca02c')
    if avg_val is not None:
        ax.plot(epochs_ran, avg_val, label='Val MAE (norm.)',  linewidth=2, color='#d62728', linestyle='--')
    ax.axvline(x=best_ep, color='green', linestyle=':', alpha=0.7)
    ax.set_title(f'Avg MAE — Infection % at {horizon} (all diseases)', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MAE (normalised, ÷100)')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
p = save_fig(fig, 'history_mae_infection')
print(f'MAE curves → {p}')
plt.show(); plt.close(fig)

# ── 17.4  Per-disease loss breakdown (inf vs act per horizon) ─
fig, axes = plt.subplots(2, N_DISEASES, figsize=(4.5 * N_DISEASES, 9), squeeze=False)
for row, horizon in enumerate(['24h', '48h']):
    for col, disease in enumerate(DISEASES):
        ax = axes[row, col]
        pairs = [
            (f'inf_{horizon}_{disease}_loss',     f'val_inf_{horizon}_{disease}_loss',     'Inf% loss', '#1f77b4', '#aec7e8'),
            (f'act_{horizon}_{disease}_loss',     f'val_act_{horizon}_{disease}_loss',     'Active loss', '#ff7f0e', '#ffbb78'),
            (f'dlt_{horizon}_{disease}_loss',     f'val_dlt_{horizon}_{disease}_loss',     'Delta loss', '#2ca02c', '#98df8a'),
        ]
        plotted = False
        for tr_key, val_key, label, tr_clr, val_clr in pairs:
            if tr_key in hist:
                ax.plot(epochs_ran, hist[tr_key], label=f'{label} (tr)', linewidth=1.5, color=tr_clr)
                plotted = True
            if val_key in hist:
                ax.plot(epochs_ran, hist[val_key], label=f'{label} (val)', linewidth=1.5, color=val_clr, linestyle='--')
        ax.axvline(x=best_ep, color='green', linestyle=':', alpha=0.5)
        disease_short = disease.replace('_', '\n')
        ax.set_title(f'{disease_short} — {horizon}', fontsize=9)
        ax.set_xlabel('Epoch', fontsize=8); ax.set_ylabel('Loss', fontsize=8)
        ax.tick_params(labelsize=7); ax.grid(True, alpha=0.3)
        if col == 0 and plotted:
            ax.legend(fontsize=6.5)

fig.suptitle('Per-Disease Loss Breakdown — Train vs Validation', fontsize=13, fontweight='bold')
plt.tight_layout()
p = save_fig(fig, 'history_per_disease_loss')
print(f'Per-disease loss plot → {p}')
plt.show(); plt.close(fig)

print('\n✅ Training history plots complete.')
logger.info('Training history plots saved.')


Total loss curve → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\history_total_loss.png
Accuracy curves → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\history_accuracy_active.png
MAE curves → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\history_mae_infection.png


2026-03-12 17:15:23,648 | INFO | Training history plots saved.


Per-disease loss plot → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\history_per_disease_loss.png

✅ Training history plots complete.


## SECTION 18 — Comprehensive Test Set Evaluation

Evaluate the trained model on the held-out test set across all 30 output heads.

**Regression heads** (infection %, net change):
- MAE, RMSE, R² per disease × horizon (24h / 48h)

**Classification heads** (active within 24h / 48h):
- Accuracy, Precision, Recall, F1-score, AUC-ROC per disease × horizon
- Full `classification_report` per disease

Results are saved to `evaluation_metrics_{RUN_ID}.json`.

In [87]:

# ─────────────────────────────────────────────────────────────
# SECTION 18: Comprehensive Test Set Evaluation
# ─────────────────────────────────────────────────────────────

# Classification threshold — lowered from 0.5 to 0.35 because:
#   1. Focal loss compresses output probabilities toward the centre (less extremes)
#   2. We prefer higher recall (catching real outbreaks) over higher precision
#   3. ROC curve analysis typically shows the optimal operating point around 0.3-0.4
#      for imbalanced or uncertain binary tasks
CLS_THR = 0.35

print('Running predictions on test set…')
preds_raw = model.predict(X_te_sc, batch_size=256, verbose=0)

# Handle single-array vs list (safety for both Keras APIs)
if not isinstance(preds_raw, list):
    preds_raw = [preds_raw]

# Build output_name → 1-D prediction array
pred_dict = {name: preds_raw[i].squeeze() for i, name in enumerate(output_names)}
print(f'Predictions collected for {len(pred_dict)} output heads.')
print(f'Classification threshold : {CLS_THR}')

# ── Metric accumulators (used again in Section 22) ───────────
eval_metrics  = {}
inf24_maes    = []
inf48_maes    = []
dlt24_maes    = []
act24_accs    = []
act48_accs    = []
act24_f1s     = []
act48_f1s     = []
act24_aucs    = []
act48_aucs    = []


def reg_metrics(actual, predicted):
    """MAE, RMSE, R² ignoring NaN positions."""
    valid = ~np.isnan(actual)
    if valid.sum() < 5:
        return None, None, None
    a, p = actual[valid], predicted[valid]
    mae  = float(mean_absolute_error(a, p))
    rmse = float(np.sqrt(mean_squared_error(a, p)))
    ss_r = np.sum((a - p) ** 2)
    ss_t = np.sum((a - a.mean()) ** 2)
    r2   = float(1 - ss_r / (ss_t + 1e-9))
    return mae, rmse, r2


def cls_metrics(actual, prob, thr=CLS_THR):
    """Acc, Prec, Recall, F1, AUC ignoring NaN positions."""
    valid = ~np.isnan(actual)
    if valid.sum() < 5 or len(np.unique(actual[valid])) < 2:
        return None, None, None, None, None
    a    = actual[valid].astype(int)
    p_b  = (prob[valid] >= thr).astype(int)
    pr_v = prob[valid]
    acc  = float(accuracy_score(a, p_b))
    prec = float(precision_score(a, p_b, zero_division=0))
    rec  = float(recall_score(a, p_b, zero_division=0))
    f1   = float(f1_score(a, p_b, zero_division=0))
    auc  = float(roc_auc_score(a, pr_v))
    return acc, prec, rec, f1, auc


# ── Header row ───────────────────────────────────────────────
SEP = '─' * 82
print(f'\n{SEP}')
print(f'{"Disease":20s}  {"Task":14s}  {"MAE":>7s}  {"RMSE":>7s}  {"R²":>7s}  {"Acc":>7s}  {"F1":>7s}  {"AUC":>7s}')
print(SEP)

for d_idx, disease in enumerate(DISEASES):

    # ── inf_24h regression ─────────────────────────────────
    p_inf24 = pred_dict[f'inf_24h_{disease}'] * INF_SCALE
    mae, rmse, r2 = reg_metrics(y_inf24_te[:, d_idx], p_inf24)
    if mae is not None:
        eval_metrics.update({f'{disease}_inf24_mae': mae, f'{disease}_inf24_rmse': rmse, f'{disease}_inf24_r2': r2})
        inf24_maes.append(mae)
        print(f'{disease:20s}  {"inf_24h":14s}  {mae:7.4f}  {rmse:7.4f}  {r2:7.4f}  {"—":>7s}  {"—":>7s}  {"—":>7s}')

    # ── inf_48h regression ─────────────────────────────────
    p_inf48 = pred_dict[f'inf_48h_{disease}'] * INF_SCALE
    mae, rmse, r2 = reg_metrics(y_inf48_te[:, d_idx], p_inf48)
    if mae is not None:
        eval_metrics.update({f'{disease}_inf48_mae': mae, f'{disease}_inf48_rmse': rmse, f'{disease}_inf48_r2': r2})
        inf48_maes.append(mae)
        print(f'{disease:20s}  {"inf_48h":14s}  {mae:7.4f}  {rmse:7.4f}  {r2:7.4f}  {"—":>7s}  {"—":>7s}  {"—":>7s}')

    # ── dlt_24h regression ─────────────────────────────────
    p_dlt24 = pred_dict[f'dlt_24h_{disease}'] * DLT_STD + DLT_MEAN
    mae, rmse, r2 = reg_metrics(y_dlt24_te[:, d_idx], p_dlt24)
    if mae is not None:
        eval_metrics.update({f'{disease}_dlt24_mae': mae, f'{disease}_dlt24_rmse': rmse, f'{disease}_dlt24_r2': r2})
        dlt24_maes.append(mae)
        print(f'{disease:20s}  {"dlt_24h":14s}  {mae:7.4f}  {rmse:7.4f}  {r2:7.4f}  {"—":>7s}  {"—":>7s}  {"—":>7s}')

    # ── dlt_48h regression ─────────────────────────────────
    p_dlt48 = pred_dict[f'dlt_48h_{disease}'] * DLT_STD + DLT_MEAN
    mae, rmse, r2 = reg_metrics(y_dlt48_te[:, d_idx], p_dlt48)
    if mae is not None:
        eval_metrics.update({f'{disease}_dlt48_mae': mae, f'{disease}_dlt48_rmse': rmse, f'{disease}_dlt48_r2': r2})
        print(f'{disease:20s}  {"dlt_48h":14s}  {mae:7.4f}  {rmse:7.4f}  {r2:7.4f}  {"—":>7s}  {"—":>7s}  {"—":>7s}')

    # ── act_24h binary classification ──────────────────────
    acc, prec, rec, f1, auc = cls_metrics(y_act24_te[:, d_idx], pred_dict[f'act_24h_{disease}'])
    if acc is not None:
        eval_metrics.update({
            f'{disease}_act24_accuracy': acc, f'{disease}_act24_precision': prec,
            f'{disease}_act24_recall': rec, f'{disease}_act24_f1': f1, f'{disease}_act24_auc': auc,
        })
        act24_accs.append(acc); act24_f1s.append(f1); act24_aucs.append(auc)
        print(f'{disease:20s}  {"act_24h":14s}  {"—":>7s}  {"—":>7s}  {"—":>7s}  {acc:7.4f}  {f1:7.4f}  {auc:7.4f}')

    # ── act_48h binary classification ──────────────────────
    acc, prec, rec, f1, auc = cls_metrics(y_act48_te[:, d_idx], pred_dict[f'act_48h_{disease}'])
    if acc is not None:
        eval_metrics.update({
            f'{disease}_act48_accuracy': acc, f'{disease}_act48_precision': prec,
            f'{disease}_act48_recall': rec, f'{disease}_act48_f1': f1, f'{disease}_act48_auc': auc,
        })
        act48_accs.append(acc); act48_f1s.append(f1); act48_aucs.append(auc)
        print(f'{disease:20s}  {"act_48h":14s}  {"—":>7s}  {"—":>7s}  {"—":>7s}  {acc:7.4f}  {f1:7.4f}  {auc:7.4f}')

    print(SEP if d_idx < N_DISEASES - 1 else '')

# ── Aggregate statistics ─────────────────────────────────────
print(f'\n{"═"*55}')
print('AGGREGATE TEST METRICS (mean across diseases)')
print(f'{"═"*55}')
agg_rows = [
    ('MAE  — Infection % 24h', inf24_maes,  '%'),
    ('MAE  — Infection % 48h', inf48_maes,  '%'),
    ('MAE  — Net change  24h', dlt24_maes,  'pp'),
    ('Acc  — Active 24h',      act24_accs,  ''),
    ('Acc  — Active 48h',      act48_accs,  ''),
    ('F1   — Active 24h',      act24_f1s,   ''),
    ('F1   — Active 48h',      act48_f1s,   ''),
    ('AUC  — Active 24h',      act24_aucs,  ''),
    ('AUC  — Active 48h',      act48_aucs,  ''),
]
for label, vals, unit in agg_rows:
    if vals:
        mu = np.mean(vals); sd = np.std(vals)
        print(f'  {label:30s}: {mu:.4f} ± {sd:.4f}  {unit}')

# ── Detailed classification reports ─────────────────────────
print(f'\n{"─"*55}')
print(f'CLASSIFICATION REPORTS — Active within 24h  (threshold={CLS_THR})')
print(f'{"─"*55}')
for d_idx, disease in enumerate(DISEASES):
    valid = ~np.isnan(y_act24_te[:, d_idx])
    if valid.sum() < 5:
        continue
    a_v = y_act24_te[valid, d_idx].astype(int)
    p_v = (pred_dict[f'act_24h_{disease}'][valid] >= CLS_THR).astype(int)
    print(f'\n  ► {disease}')
    print(classification_report(a_v, p_v, target_names=['Inactive', 'Active'], zero_division=0,
                                 digits=4))

# ── Save metrics JSON ─────────────────────────────────────────
EVAL_METRICS_PATH = os.path.join(ARTIFACTS_DIR, f'evaluation_metrics_{RUN_ID}.json')
with open(EVAL_METRICS_PATH, 'w') as f:
    json.dump(eval_metrics, f, indent=2)
print(f'Evaluation metrics saved → {EVAL_METRICS_PATH}')
logger.info(f'Evaluation complete | {len(eval_metrics)} metrics saved.')


Running predictions on test set…
Predictions collected for 30 output heads.
Classification threshold : 0.35

──────────────────────────────────────────────────────────────────────────────────
Disease               Task                MAE     RMSE       R²      Acc       F1      AUC
──────────────────────────────────────────────────────────────────────────────────
early_blight          inf_24h          5.4678  10.3411   0.8993        —        —        —
early_blight          inf_48h          8.6091  13.9817   0.8168        —        —        —
early_blight          dlt_24h          3.1256   6.4518   0.0035        —        —        —
early_blight          dlt_48h          4.8144  10.8453   0.0092        —        —        —
early_blight          act_24h               —        —        —   0.9058   0.9036   0.9745
early_blight          act_48h               —        —        —   0.9080   0.9053   0.9812
──────────────────────────────────────────────────────────────────────────────────
late_

2026-03-12 17:16:26,756 | INFO | Evaluation complete | 110 metrics saved.


              precision    recall  f1-score   support

    Inactive     0.7762    0.9955    0.8723      1348
      Active     0.9936    0.7062    0.8256      1317

    accuracy                         0.8525      2665
   macro avg     0.8849    0.8508    0.8489      2665
weighted avg     0.8836    0.8525    0.8492      2665


  ► powdery_mildew
              precision    recall  f1-score   support

    Inactive     0.9327    0.7253    0.8160      1394
      Active     0.7577    0.9426    0.8401      1271

    accuracy                         0.8289      2665
   macro avg     0.8452    0.8339    0.8280      2665
weighted avg     0.8492    0.8289    0.8275      2665


  ► spider_mites
              precision    recall  f1-score   support

    Inactive     0.8688    0.7427    0.8008      1364
      Active     0.7658    0.8824    0.8200      1301

    accuracy                         0.8109      2665
   macro avg     0.8173    0.8125    0.8104      2665
weighted avg     0.8185    0.8109   

## SECTION 19 — Confusion Matrix Heatmaps

Plot binary classification confusion matrices for all 5 diseases × 2 horizons (24h and 48h).
Each matrix shows **True Inactive / True Active** vs **Predicted Inactive / Predicted Active**.
Separate heatmap figures are saved for the 24h horizon and the 48h horizon.

In [88]:

# ─────────────────────────────────────────────────────────────
# SECTION 19: Confusion Matrix Heatmaps
# ─────────────────────────────────────────────────────────────
# Uses the same CLS_THR=0.35 as the evaluation cell so that the
# confusion matrices reflect the actual operating threshold.
# F1 values are read from eval_metrics using the correct key format
# (_act24_f1 / _act48_f1, matching the keys stored in Section 18).
# ─────────────────────────────────────────────────────────────

TARGET_LABELS = ['Inactive', 'Active']
CMAP_CM       = 'Blues'


def plot_cm_strip(y_actual_mat, pred_key_template, horizon_label, filename):
    """Plot one row of confusion matrices (1 per disease) for a given horizon."""
    h_num = horizon_label[:2]   # '24' or '48'
    fig, axes = plt.subplots(1, N_DISEASES, figsize=(4.5 * N_DISEASES, 4.5))
    fig.suptitle(
        f'Confusion Matrices — Active within {horizon_label}  '
        f'(threshold={CLS_THR})',
        fontsize=13, fontweight='bold',
    )

    for d_idx, (disease, ax) in enumerate(zip(DISEASES, axes)):
        a_act = y_actual_mat[:, d_idx]
        p_prob = pred_dict[pred_key_template.format(disease)]
        p_act  = (p_prob >= CLS_THR).astype(int)
        valid  = ~np.isnan(a_act)

        title_str = disease.replace('_', '\n').title()
        if valid.sum() < 5 or len(np.unique(a_act[valid])) < 2:
            ax.text(0.5, 0.5, 'Insufficient\ndata', ha='center', va='center',
                    fontsize=12, transform=ax.transAxes)
            ax.set_title(title_str, fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])
            continue

        cm = confusion_matrix(a_act[valid].astype(int), p_act[valid], labels=[0, 1])

        # Manual seaborn heatmap for clean formatting
        df_cm = pd.DataFrame(cm, index=TARGET_LABELS, columns=TARGET_LABELS)
        sns.heatmap(df_cm, annot=True, fmt='d', cmap=CMAP_CM, ax=ax,
                    linewidths=0.5, cbar=False,
                    annot_kws={'fontsize': 12, 'fontweight': 'bold'})
        ax.set_xlabel('Predicted', fontsize=9)
        ax.set_ylabel('Actual',    fontsize=9)

        # Metrics annotation — use the correct key format _act{h_num}_f1
        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn + 1e-9)
        specificity = tn / (tn + fp + 1e-9)
        f1_val = eval_metrics.get(f'{disease}_act{h_num}_f1', float('nan'))
        ax.set_title(
            f'{title_str}\nF1={f1_val:.3f}  Sens={sensitivity:.3f}  Spec={specificity:.3f}',
            fontsize=8.5,
        )

    plt.tight_layout()
    path = save_fig(fig, filename)
    print(f'Confusion matrix ({horizon_label}) → {path}')
    plt.show()
    plt.close(fig)


plot_cm_strip(y_act24_te, 'act_24h_{}', '24h', 'confusion_matrix_active_24h')
plot_cm_strip(y_act48_te, 'act_48h_{}', '48h', 'confusion_matrix_active_48h')

# ── Combined 2-row × N_DISEASES figure for reports ────────────
fig, axes = plt.subplots(2, N_DISEASES, figsize=(4.5 * N_DISEASES, 9), squeeze=False)
fig.suptitle(
    'Confusion Matrices — Active Outbreak Prediction\n'
    f'(top: 24h horizon | bottom: 48h horizon | threshold={CLS_THR})',
    fontsize=13, fontweight='bold',
)

for row, (y_te_bin, horizon, h_key) in enumerate(
    [(y_act24_te, '24h', '24'), (y_act48_te, '48h', '48')]
):
    for col, disease in enumerate(DISEASES):
        ax = axes[row, col]
        a_act = y_te_bin[:, col]
        p_prob = pred_dict[f'act_{horizon}_{disease}']
        p_act  = (p_prob >= CLS_THR).astype(int)
        valid  = ~np.isnan(a_act)

        title_str = disease.replace('_', '\n').title()
        if valid.sum() < 5 or len(np.unique(a_act[valid])) < 2:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title_str, fontsize=9)
            continue

        cm  = confusion_matrix(a_act[valid].astype(int), p_act[valid], labels=[0, 1])
        df_cm = pd.DataFrame(cm, index=TARGET_LABELS, columns=TARGET_LABELS)
        sns.heatmap(df_cm, annot=True, fmt='d', cmap=CMAP_CM, ax=ax,
                    linewidths=0.5, cbar=False,
                    annot_kws={'fontsize': 11, 'fontweight': 'bold'})
        ax.set_xlabel('Predicted', fontsize=8)
        ax.set_ylabel('Actual',    fontsize=8)
        # Fix: key uses _act{h_key}_f1, not _act{h_key}h_f1
        f1_val = eval_metrics.get(f'{disease}_act{h_key}_f1', float('nan'))
        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn + 1e-9)
        specificity = tn / (tn + fp + 1e-9)
        ax.set_title(
            f'{title_str} ({horizon})\n'
            f'F1={f1_val:.3f}  Sens={sensitivity:.3f}  Spec={specificity:.3f}',
            fontsize=8.5,
        )
        ax.tick_params(labelsize=8)

plt.tight_layout()
p = save_fig(fig, 'confusion_matrices_combined')
print(f'Combined confusion matrices → {p}')
plt.show()
plt.close(fig)

logger.info('Confusion matrices saved.')


Confusion matrix (24h) → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\confusion_matrix_active_24h.png
Confusion matrix (48h) → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\confusion_matrix_active_48h.png


2026-03-12 17:16:45,403 | INFO | Confusion matrices saved.


Combined confusion matrices → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\confusion_matrices_combined.png


## SECTION 20 — Prediction vs Ground Truth & Regression Diagnostics

Visual diagnostic plots for the regression outputs:

1. **Scatter plots** — Predicted vs actual infection % at 24h and 48h, colour-coded by scenario (healthy / single\_disease / multi\_disease)
2. **Residual plots** — Prediction error (predicted − actual) as a function of actual value, for detecting systematic bias
3. **Time-series preview** — Month-scale trajectory of actual vs predicted infection % for a representative test cycle, confirming the model captures disease dynamics over time

In [89]:
# ─────────────────────────────────────────────────────────────
# SECTION 20: Prediction vs Ground Truth & Regression Diagnostics
# ─────────────────────────────────────────────────────────────

SCENARIO_PALETTE = {
    'healthy':        '#2196F3',   # blue
    'single_disease': '#FF9800',   # orange
    'multi_disease':  '#F44336',   # red
}

# ── 20.1  Scatter: predicted vs actual — infection % 24h & 48h ─
for horizon, y_te_inf in [('24h', y_inf24_te), ('48h', y_inf48_te)]:
    fig, axes = plt.subplots(1, N_DISEASES, figsize=(4.5 * N_DISEASES, 4.5))
    fig.suptitle(f'Predicted vs Actual — Infection % at {horizon}',
                 fontsize=13, fontweight='bold')

    for d_idx, (disease, ax) in enumerate(zip(DISEASES, axes)):
        a_inf = y_te_inf[:, d_idx]
        p_inf = pred_dict[f'inf_{horizon}_{disease}'] * INF_SCALE
        valid = ~np.isnan(a_inf)

        if valid.sum() > 5:
            for sc_name, sc_col in SCENARIO_PALETTE.items():
                mask = valid & (sc_te == sc_name)
                if mask.sum() > 0:
                    ax.scatter(a_inf[mask], p_inf[mask],
                               alpha=0.35, s=6, color=sc_col, label=sc_name)

            # Perfect prediction diagonal
            max_v = max(float(a_inf[valid].max()), float(p_inf[valid].max()))
            ax.plot([0, max_v], [0, max_v], 'k--', linewidth=1, alpha=0.6, label='Perfect')

            mae_v = float(mean_absolute_error(a_inf[valid], p_inf[valid]))
            ax.set_title(
                f"{disease.replace('_', ' ').title()}\nMAE={mae_v:.2f}%", fontsize=9
            )

        ax.set_xlabel('Actual (%)', fontsize=8)
        ax.set_ylabel('Predicted (%)', fontsize=8)
        ax.tick_params(labelsize=7)
        if d_idx == 0 and valid.sum() > 5:
            ax.legend(fontsize=7, markerscale=2.5)

    plt.tight_layout()
    p = save_fig(fig, f'scatter_infection_pct_{horizon}')
    print(f'Scatter plot ({horizon}) → {p}')
    plt.show()
    plt.close(fig)

# ── 20.2  Residual plots — infection % (bias detection) ───────
fig, axes = plt.subplots(2, N_DISEASES, figsize=(4.5 * N_DISEASES, 9), squeeze=False)
fig.suptitle('Residual Analysis — Infection % Predictions\n(Predicted − Actual)',
             fontsize=13, fontweight='bold')

for row, (horizon, y_te_inf) in enumerate([('24h', y_inf24_te), ('48h', y_inf48_te)]):
    for d_idx, (disease, ax) in enumerate(zip(DISEASES, axes[row])):
        a_inf = y_te_inf[:, d_idx]
        p_inf = pred_dict[f'inf_{horizon}_{disease}'] * INF_SCALE
        valid = ~np.isnan(a_inf)

        if valid.sum() > 5:
            residuals = p_inf[valid] - a_inf[valid]
            ax.scatter(a_inf[valid], residuals, alpha=0.3, s=6, color='steelblue')
            ax.axhline(0, color='red', linestyle='--', linewidth=1.2, label='Zero error')
            # Loess-style smoothing using rolling mean on sorted data
            sort_idx = np.argsort(a_inf[valid])
            a_sorted = a_inf[valid][sort_idx]
            r_sorted = residuals[sort_idx]
            window   = max(5, len(r_sorted) // 20)
            r_smooth = pd.Series(r_sorted).rolling(window, min_periods=1, center=True).mean().values
            ax.plot(a_sorted, r_smooth, color='darkorange', linewidth=1.5, label='Trend')
            ax.legend(fontsize=7)

        disease_short = disease.replace('_', ' ').title()
        ax.set_title(f'{disease_short} — {horizon}', fontsize=9)
        ax.set_xlabel('Actual (%)', fontsize=8)
        ax.set_ylabel('Residual (Pred−Actual, %)', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
p = save_fig(fig, 'residual_plots_infection_pct')
print(f'Residual plots → {p}')
plt.show()
plt.close(fig)

# ── 20.3  Time-series preview for an active test cycle ─────────
test_cycle_ids = cycle_ids_all[test_mask]
# Pick the test cycle with the highest max infection
cycle_peak_inf = {
    c: float(y_inf24_te[test_cycle_ids == c, :].max())
    for c in np.unique(test_cycle_ids)
}
showcase_cycle = max(cycle_peak_inf, key=cycle_peak_inf.get)
cyc_mask       = test_cycle_ids == showcase_cycle

print(f'\nTime-series preview cycle: {showcase_cycle}  '
      f'(peak infection ≈ {cycle_peak_inf[showcase_cycle]:.1f}%,  {int(cyc_mask.sum())} samples)')

fig, axes = plt.subplots(N_DISEASES, 2, figsize=(15, 4 * N_DISEASES), squeeze=False)
fig.suptitle(f'Actual vs Predicted Infection % — Test Cycle {showcase_cycle}',
             fontsize=13, fontweight='bold')

for d_idx, disease in enumerate(DISEASES):
    for col, (horizon, y_te_inf) in enumerate([('24h', y_inf24_te), ('48h', y_inf48_te)]):
        ax      = axes[d_idx, col]
        a_inf   = y_te_inf[cyc_mask, d_idx]
        p_inf   = pred_dict[f'inf_{horizon}_{disease}'][cyc_mask] * INF_SCALE
        valid   = ~np.isnan(a_inf)
        t_steps = np.arange(int(cyc_mask.sum()))

        if valid.sum() > 2:
            ax.fill_between(t_steps[valid], a_inf[valid], alpha=0.15, color='#1f77b4')
            ax.plot(t_steps[valid], a_inf[valid], color='#1f77b4',
                    linewidth=1.5, label='Actual')
            ax.plot(t_steps[valid], p_inf[valid], color='#ff7f0e',
                    linewidth=1.5, linestyle='--', label='Predicted')
            ax.axhline(0, color='grey', linewidth=0.5, linestyle=':')

        ax.set_title(f"{disease.replace('_', ' ').title()} — {horizon}", fontsize=9)
        ax.set_xlabel('Time step (h)', fontsize=8)
        ax.set_ylabel('Infection %', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(True, alpha=0.3)
        if d_idx == 0 and col == 0:
            ax.legend(fontsize=8)

plt.tight_layout()
p = save_fig(fig, f'timeseries_preview_cycle_{showcase_cycle}')
print(f'Time-series preview → {p}')
plt.show()
plt.close(fig)

logger.info(f'Regression diagnostic plots saved. Showcase cycle={showcase_cycle}.')


Scatter plot (24h) → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\scatter_infection_pct_24h.png
Scatter plot (48h) → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\scatter_infection_pct_48h.png
Residual plots → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\residual_plots_infection_pct.png

Time-series preview cycle: 1  (peak infection ≈ nan%,  2689 samples)


2026-03-12 17:17:17,184 | INFO | Regression diagnostic plots saved. Showcase cycle=1.


Time-series preview → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\timeseries_preview_cycle_1.png


## SECTION 21 — Feature Importance

Two complementary approaches to model interpretability:

**21.1 — Integrated Gradient Importance (always available)**
Computes the mean absolute gradient of the total output magnitude w.r.t. each input feature (averaged over time-steps and samples). This gives a model-based ranking of which features the GRU network uses most.

**21.2 — SHAP DeepExplainer (if `shap` is installed)**
Uses SHAP DeepExplainer with a background sample of training data to compute Shapley values for the primary regression head (`inf_24h_{disease_0}`). Produces both a bar chart and a beeswarm plot.

In [90]:
# ─────────────────────────────────────────────────────────────
# SECTION 21: Feature Importance
# ─────────────────────────────────────────────────────────────

TOP_K = 20

# ── 21.1  Integrated Gradient Importance ─────────────────────
# Average absolute gradient of the model output w.r.t. input features.
# Aggregated over all output heads and a background sample of test data.

print('Computing gradient-based feature importance…')
BG_N   = min(512, X_te_sc.shape[0])
bg_idx = np.random.default_rng(RANDOM_SEED).choice(X_te_sc.shape[0], BG_N, replace=False)
X_bg   = tf.constant(X_te_sc[bg_idx].astype(np.float32))


@tf.function
def batch_gradients(x_batch):
    """Sum-absolute gradient over all heads for a single batch."""
    with tf.GradientTape() as tape:
        tape.watch(x_batch)
        outputs = model(x_batch, training=False)
        # Aggregate: mean absolute activation across all output heads
        if isinstance(outputs, (list, tuple)):
            agg = tf.reduce_mean(tf.stack([tf.abs(o) for o in outputs], axis=-1), axis=-1)
        else:
            agg = tf.reduce_mean(tf.abs(outputs), axis=-1)
        agg_scalar = tf.reduce_mean(agg)
    return tape.gradient(agg_scalar, x_batch)


GRAD_BATCH = 64
all_grads = []
for start in range(0, BG_N, GRAD_BATCH):
    x_chunk = X_bg[start: start + GRAD_BATCH]
    g = batch_gradients(x_chunk)
    if g is not None:
        all_grads.append(np.abs(g.numpy()))

if all_grads:
    grad_arr = np.concatenate(all_grads, axis=0)          # (N, SEQ_LEN, N_feat)
    feat_imp_grad = grad_arr.mean(axis=(0, 1))             # (N_feat,)

    feat_imp_series = pd.Series(feat_imp_grad, index=WIDE_FEATURE_COLS).sort_values(ascending=False)
    top_feats       = feat_imp_series.head(TOP_K)

    fig, ax = plt.subplots(figsize=(10, 7))
    clrs = plt.cm.viridis(np.linspace(0.2, 0.85, TOP_K))[::-1]
    top_feats.sort_values().plot(kind='barh', ax=ax, color=clrs)
    ax.set_title(f'Top-{TOP_K} Features — Integrated Gradient Importance\n(aggregated over all 30 output heads)',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Mean |∂Output/∂Input|', fontsize=10)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    p = save_fig(fig, 'gradient_feature_importance')
    print(f'Gradient importance plot → {p}')
    plt.show()
    plt.close(fig)

    # Save CSV
    feat_imp_df = pd.DataFrame({
        'feature'    : feat_imp_series.index,
        'importance' : feat_imp_series.values,
        'rank'       : range(1, len(feat_imp_series) + 1),
    })
    FI_CSV = os.path.join(ARTIFACTS_DIR, 'gradient_feature_importance.csv')
    feat_imp_df.to_csv(FI_CSV, index=False)
    print(f'Gradient importance CSV → {FI_CSV}')
    logger.info(f'Gradient feature importance saved. Top feature: {feat_imp_series.index[0]}')
else:
    print('⚠️  Gradient computation returned empty — skipping gradient importance.')

# ── 21.2  Per-disease gradient importance ────────────────────
# For each disease, compute gradient through just its inf_24h head.
print('\nComputing per-disease gradient importance (inf_24h)…')

fig, axes = plt.subplots(1, N_DISEASES, figsize=(5 * N_DISEASES, 6), squeeze=False)
fig.suptitle(f'Top-15 Feature Importance per Disease (Gradient, inf_24h)',
             fontsize=13, fontweight='bold')

per_disease_fi = {}
for d_idx, disease in enumerate(DISEASES):
    head_name = f'inf_24h_{disease}'
    try:
        sub_model = keras.Model(inputs=model.input,
                                outputs=model.get_layer(head_name).output)
    except ValueError:
        axes[0, d_idx].set_title(f'{disease}\n(layer not found)', fontsize=8)
        continue

    @tf.function
    def disease_gradients(x_b, sm=sub_model):
        with tf.GradientTape() as t:
            t.watch(x_b)
            out = sm(x_b, training=False)
        return t.gradient(tf.reduce_mean(tf.abs(out)), x_b)

    d_grads = []
    for start in range(0, BG_N, GRAD_BATCH):
        x_chunk = X_bg[start: start + GRAD_BATCH]
        g = disease_gradients(x_chunk)
        if g is not None:
            d_grads.append(np.abs(g.numpy()))

    if d_grads:
        d_arr = np.concatenate(d_grads, axis=0)
        d_imp = pd.Series(d_arr.mean(axis=(0, 1)), index=WIDE_FEATURE_COLS).sort_values(ascending=False)
        per_disease_fi[disease] = d_imp
        top_d = d_imp.head(15)
        ax = axes[0, d_idx]
        top_d.sort_values().plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
        ax.set_title(f"{disease.replace('_', ' ').title()}", fontsize=9, fontweight='bold')
        ax.set_xlabel('|Gradient|', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
p = save_fig(fig, 'gradient_feature_importance_per_disease')
print(f'Per-disease importance plot → {p}')
plt.show()
plt.close(fig)

# ── 21.3  SHAP (if available) ─────────────────────────────────
if SHAP_AVAILABLE:
    print('\nComputing SHAP explanations (DeepExplainer)…')
    SHAP_BG_N   = min(100, X_tr_sc.shape[0])
    SHAP_EXPL_N = min(200, X_te_sc.shape[0])
    bg_shap  = X_tr_sc[np.random.default_rng(RANDOM_SEED + 1).choice(X_tr_sc.shape[0],  SHAP_BG_N,   replace=False)].astype(np.float32)
    ex_shap  = X_te_sc[np.random.default_rng(RANDOM_SEED + 2).choice(X_te_sc.shape[0], SHAP_EXPL_N, replace=False)].astype(np.float32)

    try:
        shap_head = f'inf_24h_{DISEASES[0]}'
        shap_sub  = keras.Model(inputs=model.input,
                                outputs=model.get_layer(shap_head).output)
        explainer   = shap.DeepExplainer(shap_sub, bg_shap)
        shap_values = explainer.shap_values(ex_shap)

        # Mean abs SHAP across samples and time-steps
        shap_arr = np.abs(shap_values[0] if isinstance(shap_values, list) else shap_values)
        shap_mean = shap_arr.mean(axis=(0, 1))   # (N_feat,)
        shap_imp  = pd.Series(shap_mean, index=WIDE_FEATURE_COLS).sort_values(ascending=False)
        top_shap  = shap_imp.head(TOP_K)

        fig, ax = plt.subplots(figsize=(10, 7))
        top_shap.sort_values().plot(kind='barh', ax=ax, color='#d62728', alpha=0.85)
        ax.set_title(f'Top-{TOP_K} SHAP Feature Importance — {shap_head}',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Mean |SHAP value|', fontsize=10)
        ax.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        p = save_fig(fig, 'shap_feature_importance')
        print(f'SHAP importance plot → {p}')
        plt.show()
        plt.close(fig)

        shap_df = pd.DataFrame({'feature': shap_imp.index, 'shap_mean_abs': shap_imp.values})
        SHAP_CSV = os.path.join(ARTIFACTS_DIR, 'shap_feature_importance.csv')
        shap_df.to_csv(SHAP_CSV, index=False)
        print(f'SHAP importance CSV → {SHAP_CSV}')
        logger.info(f'SHAP importance saved. Top feature: {shap_imp.index[0]}')

    except Exception as exc:
        print(f'SHAP computation failed: {exc}')
        print('  Continuing without SHAP.')
else:
    print('\nSHAP not installed — skipping SHAP plots.')
    print('  Install with:  pip install shap')

print('\n✅ Feature importance section complete.')


Computing gradient-based feature importance…
Gradient importance plot → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\gradient_feature_importance.png


2026-03-12 17:17:58,687 | INFO | Gradient feature importance saved. Top feature: day_night_flag_roll_std_24h


Gradient importance CSV → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\gradient_feature_importance.csv

Computing per-disease gradient importance (inf_24h)…
Per-disease importance plot → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\gradient_feature_importance_per_disease.png

Computing SHAP explanations (DeepExplainer)…
SHAP computation failed: in user code:

    File "e:\AgriTwin-GH\.venv\Lib\site-packages\shap\explainers\_deep\deep_tf.py", line 265, in grad_graph  *
        x_grad = tape.gradient(out, shap_rAnD)

    LookupError: gradient registry has no entry for: shap_Neg

  Continuing without SHAP.

✅ Feature importance section complete.


## SECTION 22 — Final Artifact Export & Run Summary

Save all remaining artifacts to the run's artifact directory and print the final run report.

**Saved outputs:**
| File | Description |
|---|---|
| `disease_progression_{RUN_ID}.keras` | Full trained Keras model |
| `best_model_{RUN_ID}.keras` | Best checkpoint (lowest val\_loss) |
| `training_history_{RUN_ID}.json` | Per-epoch metrics |
| `training_log_{RUN_ID}.csv` | CSV epoch log (CSVLogger) |
| `evaluation_metrics_{RUN_ID}.json` | All test metrics |
| `sample_predictions_{RUN_ID}.csv` | 500 sample test predictions |
| `run_summary_{RUN_ID}.json` | Complete run manifest |
| `final_evaluation_summary.png` | Bar-chart summary of all metrics |
| `feature_scaler.pkl` | StandardScaler for inference |
| `inference_config.json` | Full inference configuration |

In [91]:
# ─────────────────────────────────────────────────────────────
# SECTION 22: Final Artifact Export & Run Summary
# ─────────────────────────────────────────────────────────────

# ── 22.1  Save trained Keras model ────────────────────────────
model.save(MODEL_PATH)
print(f'✅ Trained model saved → {MODEL_PATH}')
logger.info(f'Model saved → {MODEL_PATH}')

# ── 22.2  Save 500 sample test predictions ────────────────────
N_SAMPLE  = min(500, len(X_te_sc))
samp_idx  = np.random.default_rng(RANDOM_SEED).choice(len(X_te_sc), N_SAMPLE, replace=False)
sample_rows = []
for i in samp_idx:
    row = {'sample_idx': int(i), 'scenario': sc_te[i]}
    for d_idx, disease in enumerate(DISEASES):
        # Raw actual values (may be NaN at boundaries)
        a_inf24 = y_inf24_te[i, d_idx]
        a_inf48 = y_inf48_te[i, d_idx]
        a_act24 = y_act24_te[i, d_idx]
        a_dlt24 = y_dlt24_te[i, d_idx]

        row[f'{disease}_actual_inf24h']  = float(a_inf24) if not np.isnan(a_inf24) else None
        row[f'{disease}_pred_inf24h']    = float(pred_dict[f'inf_24h_{disease}'][i] * INF_SCALE)
        row[f'{disease}_actual_inf48h']  = float(a_inf48) if not np.isnan(a_inf48) else None
        row[f'{disease}_pred_inf48h']    = float(pred_dict[f'inf_48h_{disease}'][i] * INF_SCALE)
        row[f'{disease}_actual_act24h']  = float(a_act24) if not np.isnan(a_act24) else None
        row[f'{disease}_pred_act24h_prob']  = float(pred_dict[f'act_24h_{disease}'][i])
        row[f'{disease}_pred_act24h_label'] = int(pred_dict[f'act_24h_{disease}'][i] >= 0.5)
        row[f'{disease}_actual_dlt24h']  = float(a_dlt24) if not np.isnan(a_dlt24) else None
        row[f'{disease}_pred_dlt24h']    = float(pred_dict[f'dlt_24h_{disease}'][i] * DLT_STD + DLT_MEAN)
    sample_rows.append(row)

sample_df      = pd.DataFrame(sample_rows)
SAMPLE_PRED_PATH = os.path.join(ARTIFACTS_DIR, f'sample_predictions_{RUN_ID}.csv')
sample_df.to_csv(SAMPLE_PRED_PATH, index=False)
print(f'Sample predictions ({N_SAMPLE} rows) → {SAMPLE_PRED_PATH}')

# ── 22.3  Final evaluation summary bar chart ──────────────────
diseases_disp = [d.replace('_', '\n') for d in DISEASES]
x_pos         = np.arange(N_DISEASES)

metric_panels = [
    (inf24_maes,  'MAE — Infection % 24h',   'coral',         '% pts'),
    (inf48_maes,  'MAE — Infection % 48h',   'tomato',        '% pts'),
    (act24_accs,  'Accuracy — Active 24h',   'steelblue',     'fraction'),
    (act48_accs,  'Accuracy — Active 48h',   'royalblue',     'fraction'),
    (act24_f1s,   'F1 — Active 24h',         'mediumseagreen','score'),
    (act48_f1s,   'F1 — Active 48h',         'seagreen',      'score'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Final Test Set Evaluation — Disease Progression Model',
             fontsize=14, fontweight='bold')

for ax, (values, title, color, ylabel) in zip(axes.flat, metric_panels):
    # Pad / align values with diseases
    disease_vals  = [eval_metrics.get(
        f'{d}_{"inf24" if "24h" in title and "Inf" in title else "inf48" if "48h" in title and "Inf" in title else "act24" if "24h" in title and "Acc" in title or "24h" in title and "F1" in title else "act48"}_{"mae" if "MAE" in title else "accuracy" if "Acc" in title else "f1"}',
        np.nan) for d in DISEASES]

    bars = ax.bar(x_pos, disease_vals, color=color, alpha=0.82,
                  edgecolor='black', linewidth=0.5)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(diseases_disp, fontsize=8)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, disease_vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.003,
                    f'{val:.3f}', ha='center', va='bottom',
                    fontsize=8, fontweight='bold')

plt.tight_layout()
p = save_fig(fig, 'final_evaluation_summary')
print(f'Final evaluation chart → {p}')
plt.show()
plt.close(fig)

# ── 22.4  Complete run summary JSON ───────────────────────────
run_summary = {
    'run_id'             : RUN_ID,
    'architecture'       : 'MultiStreamGRU_CrossDiseaseAttention',
    'total_params'       : int(model.count_params()),
    'training_epochs'    : int(epochs_run),
    'best_val_loss'      : float(best_val),
    'final_train_loss'   : float(final_tr),
    'train_samples'      : int(X_tr_sc.shape[0]),
    'val_samples'        : int(X_va_sc.shape[0]),
    'test_samples'       : int(X_te_sc.shape[0]),
    'seq_len'            : int(SEQ_LEN),
    'n_features'         : int(N_TOTAL_FEAT),
    'n_diseases'         : int(N_DISEASES),
    'disease_names'      : DISEASES,
    'horizons_h'         : [int(HORIZON_24), int(HORIZON_48)],
    'aggregate_metrics'  : {
        'mean_mae_inf_24h'   : float(np.mean(inf24_maes))  if inf24_maes  else None,
        'mean_mae_inf_48h'   : float(np.mean(inf48_maes))  if inf48_maes  else None,
        'mean_mae_dlt_24h'   : float(np.mean(dlt24_maes))  if dlt24_maes  else None,
        'mean_acc_act_24h'   : float(np.mean(act24_accs))  if act24_accs  else None,
        'mean_acc_act_48h'   : float(np.mean(act48_accs))  if act48_accs  else None,
        'mean_f1_act_24h'    : float(np.mean(act24_f1s))   if act24_f1s   else None,
        'mean_f1_act_48h'    : float(np.mean(act48_f1s))   if act48_f1s   else None,
        'mean_auc_act_24h'   : float(np.mean(act24_aucs))  if act24_aucs  else None,
        'mean_auc_act_48h'   : float(np.mean(act48_aucs))  if act48_aucs  else None,
    },
    'per_disease_metrics': eval_metrics,
    'artifact_paths': {
        'model'               : MODEL_PATH,
        'checkpoint'          : CKPT_PATH,
        'training_history'    : history_path,
        'training_log_csv'    : CSV_LOG_PATH,
        'eval_metrics_json'   : EVAL_METRICS_PATH,
        'sample_predictions'  : SAMPLE_PRED_PATH,
        'feature_scaler_pkl'  : os.path.join(ARTIFACTS_DIR, 'feature_scaler.pkl'),
        'model_config_json'   : os.path.join(METRICS_DIR,   'model_config.json'),
        'inference_config'    : os.path.join(ARTIFACTS_DIR, 'inference_config.json'),
        'plots_dir'           : PLOTS_DIR,
    },
}

SUMMARY_PATH = os.path.join(ARTIFACTS_DIR, f'run_summary_{RUN_ID}.json')
with open(SUMMARY_PATH, 'w') as f:
    json.dump(run_summary, f, indent=2)
print(f'Run summary JSON → {SUMMARY_PATH}')

# ── 22.5  Final console summary ───────────────────────────────
print(f'\n{"═"*70}')
print(f'  DISEASE PROGRESSION MODEL — RUN COMPLETE')
print(f'{"═"*70}')
print(f'  Run ID            : {RUN_ID}')
print(f'  Architecture      : MultiStreamGRU + CrossDiseaseAttention')
print(f'  Parameters        : {model.count_params():,}')
print(f'  Epochs trained    : {epochs_run}')
print(f'  Best val loss     : {best_val:.6f}')
print(f'\n  ── Test Performance ──')
if inf24_maes:  print(f'  MAE  infection 24h : {np.mean(inf24_maes):.4f} % pts')
if inf48_maes:  print(f'  MAE  infection 48h : {np.mean(inf48_maes):.4f} % pts')
if act24_accs:  print(f'  Acc  active    24h : {np.mean(act24_accs):.4f}')
if act48_accs:  print(f'  Acc  active    48h : {np.mean(act48_accs):.4f}')
if act24_f1s:   print(f'  F1   active    24h : {np.mean(act24_f1s ):.4f}')
if act48_f1s:   print(f'  F1   active    48h : {np.mean(act48_f1s ):.4f}')
if act24_aucs:  print(f'  AUC  active    24h : {np.mean(act24_aucs):.4f}')
if act48_aucs:  print(f'  AUC  active    48h : {np.mean(act48_aucs):.4f}')
print(f'\n  Artifacts dir     : {ARTIFACTS_DIR}')
print(f'{"═"*70}')

logger.info(
    f'Run {RUN_ID} complete | '
    f'mae_inf24={np.mean(inf24_maes):.4f} f1_act24={np.mean(act24_f1s):.4f} | '
    f'artifacts → {ARTIFACTS_DIR}'
)


2026-03-12 17:18:49,135 | INFO | Model saved → E:\AgriTwin-GH\src\agritwin_gh\models\disease_progression_20260312_141842.keras


✅ Trained model saved → E:\AgriTwin-GH\src\agritwin_gh\models\disease_progression_20260312_141842.keras
Sample predictions (500 rows) → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\sample_predictions_20260312_141842.csv


2026-03-12 17:18:51,761 | INFO | Run 20260312_141842 complete | mae_inf24=7.1795 f1_act24=0.8555 | artifacts → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842


Final evaluation chart → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\final_evaluation_summary.png
Run summary JSON → E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\disease_progression_20260312_141842\run_summary_20260312_141842.json

══════════════════════════════════════════════════════════════════════
  DISEASE PROGRESSION MODEL — RUN COMPLETE
══════════════════════════════════════════════════════════════════════
  Run ID            : 20260312_141842
  Architecture      : MultiStreamGRU + CrossDiseaseAttention
  Parameters        : 181,790
  Epochs trained    : 36
  Best val loss     : 6.663090

  ── Test Performance ──
  MAE  infection 24h : 7.1795 % pts
  MAE  infection 48h : 7.6585 % pts
  Acc  active    24h : 0.8637
  Acc  active    48h : 0.8535
  F1   active    24h : 0.8555
  F1   active    48h : 0.8449
  AUC  active    24h : 0.9544
  AUC  active    48h : 0.9413

  Artifacts dir     : E:\AgriTwin-GH\src\agritwin_gh\models\artifacts\diseas